# Notebook 04: WHOISDS Ingestion and Live Scoring

This notebook implements the live WHOISDS scoring workflow. It loads and normalises WHOISDS domain batches, applies blacklist and contextual filtering, enriches selected candidates with DNS, TLS, and crawler evidence, aligns live features to the saved model schema, scores domains using the selected XGBoost model, applies a triage logic, and exports analyst review files.

## 1. Configuration

Data paths, saved model artefacts, selected thresholds, and runtime settings are loaded before live WHOISDS processing begins.

In [1]:
from __future__ import annotations
import json
import math
import os
import re
import shutil
import socket
import ssl
import time
import warnings
import zipfile
from collections import Counter
from datetime import datetime, timezone
from html import unescape
from pathlib import Path
from typing import Any, Iterable, Optional
from urllib.parse import urlparse
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import shap
from bs4 import BeautifulSoup
import dns.exception
import dns.resolver
import tldextract

# Regex patterns used to clean domain strings
_DOMAIN_PROTOCOL_RE = re.compile(r"^[a-zA-Z][a-zA-Z0-9+\-.]*://")
_DOMAIN_WHITESPACE_RE = re.compile(r"\s+")

# Public suffix extractor used to identify registered domains
_TLD_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())

# Parse date-like values safely into UTC-normalised pandas timestamps
def parse_datetime_safe(value: Any) -> pd.Timestamp:
    # Return a missing timestamp for null or NaN inputs
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return pd.NaT
    # Try to parse the value as a UTC timestamp
    try:
        timestamp = pd.to_datetime(value, errors="coerce", utc=True)
    except Exception:
        return pd.NaT
    # Return a missing timestamp when parsing fails
    if pd.isna(timestamp):
        return pd.NaT
    # Convert to UTC and remove timezone information for consistent comparisons
    return timestamp.tz_convert("UTC").tz_localize(None)

# Normalise raw domain or URL text into a clean lowercase domain 
def normalise_domain(value: Any) -> Optional[str]:
    # Return None for null or NaN inputs
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    # Convert the input to lowercase stripped text
    text = str(value).strip().lower()
    # Return None for empty strings
    if not text:
        return None
    # Remove protocols, paths, query strings, fragments, credentials, and ports
    text = _DOMAIN_PROTOCOL_RE.sub("", text)
    text = text.split("/", 1)[0].split("?", 1)[0].split("#", 1)[0]
    if "@" in text:
        text = text.rsplit("@", 1)[-1]
    text = text.split(":", 1)[0]
    # Remove edge dots and common www prefix
    text = text.lstrip(".").rstrip(".")
    if text.startswith("www."):
        text = text[4:]
    # Remove embedded whitespace
    text = _DOMAIN_WHITESPACE_RE.sub("", text)
    # Return the cleaned domain when available
    return text or None

# Extract the registered domain using tldextract after normalisation
def get_registered_domain(domain: Any) -> str:
    normalised = normalise_domain(domain) or ""
    # Return an empty string when no usable domain is available
    if not normalised:
        return ""
    # Extract subdomain, domain, and suffix components
    extracted = _TLD_EXTRACTOR(normalised)
    # Fall back to the normalised value when extraction is incomplete
    if not extracted.domain or not extracted.suffix:
        return normalised
    # Return the registered domain as domain plus public suffix
    return f"{extracted.domain}.{extracted.suffix}"

# Compute domain age and registration duration features
def compute_domain_lifecycle_features(
    df: pd.DataFrame,
    observed_col: str = "observed_at",
    registration_col: str = "registration_date",
    expiration_col: str = "expiration_date",
) -> pd.DataFrame:
    # Worked on a copy to preserve the original dataframe
    enriched = df.copy()
    # Parse observed, registration, and expiration dates as UTC timestamps
    observed_dt = pd.to_datetime(enriched[observed_col], errors="coerce", utc=True)
    registration_dt = pd.to_datetime(enriched[registration_col], errors="coerce", utc=True)
    expiration_dt = pd.to_datetime(enriched[expiration_col], errors="coerce", utc=True)
    # Calculate lifecycle durations in days
    enriched["domain_age_days"] = (observed_dt - registration_dt).dt.days
    enriched["registration_period_days"] = (expiration_dt - registration_dt).dt.days
    # Replace impossible negative lifecycle values with missing values
    enriched.loc[enriched["domain_age_days"] < 0, "domain_age_days"] = np.nan
    enriched.loc[
        enriched["registration_period_days"] < 0,
        "registration_period_days",
    ] = np.nan
    # Return the enriched dataframe
    return enriched

# Resolve the project data root from an environment variable
def resolve_data_root(
    env_var: str = "PHISHING_PROJECT_DATA_ROOT",
    preferred: Optional[Iterable[Path]] = None,
) -> Path:
    # Read the optional data root environment variable
    env_value_raw = os.environ.get(env_var, "").strip()
    env_path = Path(env_value_raw) if env_value_raw else None
    # Build the ordered list of candidate data roots
    candidates: list[Path] = []
    if env_path is not None:
        candidates.append(env_path)
    if preferred is not None:
        candidates.extend(preferred)
    candidates.extend([Path("/datasets"), Path("datasets"), Path(".")])
    # Return the first existing candidate path
    for candidate in candidates:
        try:
            if candidate.exists():
                return candidate
        except Exception:
            continue
    return Path(".")

# Widen dataframe display settings
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

# Define supported WHOISDS file and archive formats
SUPPORTED_WHOISDS_EXTENSIONS = {".csv", ".xlsx", ".xls"}
SUPPORTED_WHOISDS_ARCHIVE_EXTENSIONS = {".zip"}
SUPPORTED_WHOISDS_SOURCE_EXTENSIONS = SUPPORTED_WHOISDS_EXTENSIONS | SUPPORTED_WHOISDS_ARCHIVE_EXTENSIONS

# Resolve the project data root and WHOISDS working directories
SEARCH_ROOT_CANDIDATES = [
    Path("/datasets"),
    Path("datasets"),
    Path("."),
]

# Define live enrichment timeouts and scoring runtime settings
DATA_ROOT = resolve_data_root(preferred=SEARCH_ROOT_CANDIDATES)
WHOISDS_RAW_DIR = DATA_ROOT / "whoisds-raw"
WHOISDS_PROCESSED_DIR = DATA_ROOT / "whoisds-processed"

# Define live enrichment timeouts and scoring runtime settings
DNS_TIMEOUT = 3.0
HTTP_TIMEOUT = 6.0
TLS_TIMEOUT = 5.0
USER_AGENT = "Mozilla/5.0 (Linux; Android 14; SM-S918B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.6422.165 Mobile Safari/537.36"
DEFAULT_THRESHOLD = 0.50
TLS_PORT = 443
MAX_HTTP_REDIRECTS = 5

# Capture the live crawl timestamp in UTC
CRAWLED_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Ensure the processed WHOISDS directory exists
WHOISDS_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Store the de-duplicated search roots used by the notebook
SEARCH_ROOTS_USED = [DATA_ROOT] + SEARCH_ROOT_CANDIDATES

# Store the de-duplicated search roots used by the notebook
print(f"WHOISDS config: {DATA_ROOT.resolve()}")

WHOISDS config: /datasets


## 2. Data Loading

Saved model artefacts are loaded together with the latest WHOISDS batch selected for live scoring.

In [2]:
# Load a JSON file as a Python dictionary
def load_json(path: Path) -> dict[str, Any]:
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

# Expand relative artefact paths across all configured search roots
def expand_candidate_paths(relative_candidates: list[str]) -> list[Path]:
    # Collect expanded path candidates
    expanded = []
    # Combine each search root with each relative candidate path
    for root in SEARCH_ROOTS_USED:
        for relative_candidate in relative_candidates:
            expanded.append(root / relative_candidate)
    # Return de-duplicated candidate paths
    return expanded

# Return the first existing path from a candidate list
def resolve_first_existing(candidates: list[Path], required: bool = False, label: str = "artefact") -> Optional[Path]:
    # Search candidates in priority order
    for candidate in candidates:
        if candidate.exists():
            return candidate
    # Raise an error when a required artefact is missing
    if required:
        checked = ", ".join(str(candidate) for candidate in candidates)
        raise FileNotFoundError(f"Missing required {label}. Checked: {checked}")
    # Return the preferred fallback candidate when the artefact is optional
    return candidates[0] if candidates else None

# Determine the final selected XGBoost model name from saved metadata
def parse_final_model_name(threshold_summary: dict[str, Any], feature_metadata: dict[str, Any]) -> str:
    candidate = str(
        threshold_summary.get("model")
        or feature_metadata.get("final_model_name")
        or "XGB_tuned"
    )
    return candidate if candidate in {"XGB", "XGB_tuned"} else "XGB_tuned"

# Resolve the threshold for the selected XGBoost model
def resolve_xgboost_threshold(
    model_name: str,
    threshold_summary: dict[str, Any],
    threshold_summary_path: Path,
) -> tuple[float, str, bool]:
    if str(threshold_summary.get("model", "")) == model_name and "final_threshold" in threshold_summary:
        return float(threshold_summary.get("final_threshold", DEFAULT_THRESHOLD)), str(threshold_summary.get("final_threshold_type", "notebook3_final")), False
    # Look for the validation threshold exported by Notebook 2 for the selected XGBoost model
    threshold_csv_path = resolve_first_existing(
        expand_candidate_paths([
            "evaluation/notebook2_threshold_summary.csv",
            "notebook2_threshold_summary.csv",
        ]),
        required=False,
        label="Notebook 2 threshold summary",
    )
    if threshold_csv_path and threshold_csv_path.exists():
        threshold_df = pd.read_csv(threshold_csv_path)
        match = threshold_df.loc[threshold_df["model"].astype(str).eq(model_name)]
        if not match.empty:
            return float(match.iloc[0]["threshold"]), "notebook2_validation_threshold", False
    print("No XGBoost threshold is available.")
    return
    
# Resolve the Notebook 3 best-model manifest path
manifest_path = resolve_first_existing(
    expand_candidate_paths([
        "outputs/notebook3_best_model_manifest.json",
        "notebook3_best_model_manifest.json",
    ]),
    required=False,
    label="Notebook 3 best-model manifest",
)

# Load the best-model manifest
best_model_manifest = load_json(manifest_path) if manifest_path and manifest_path.exists() else {}

# Resolve the compact model feature schema path
model_feature_schema_path = resolve_first_existing(
    expand_candidate_paths([
        str(best_model_manifest.get("model_feature_schema_path", "models/model_feature_schema.json")),
        "models/model_feature_schema.json",
        "model_feature_schema.json",
    ]),
    required=True,
    label="model feature schema",
)

# Resolve the extended feature metadata path
feature_schema_path = resolve_first_existing(
    expand_candidate_paths([
        str(best_model_manifest.get("feature_names_path", "models/feature_names.json")),
        "models/feature_names.json",
        "feature_names.json",
    ]),
    required=True,
    label="feature schema",
)

# Resolve the final threshold summary path
threshold_summary_path = resolve_first_existing(
    expand_candidate_paths([
        str(best_model_manifest.get("threshold_summary_path", "outputs/notebook3_threshold_final_summary.json")),
        "outputs/notebook3_threshold_final_summary.json",
        "notebook3_threshold_final_summary.json",
        "outputs/threshold_summary.json",
        "threshold_summary.json",
    ]),
    required=True,
    label="threshold summary",
)

# Resolve the preprocessing artefact path when available
preprocessor_path = resolve_first_existing(
    expand_candidate_paths([
        str(best_model_manifest.get("preprocessor_path", "models/preprocessor.joblib")),
        "models/preprocessor.joblib",
        "preprocessor.joblib",
    ]),
    required=False,
    label="preprocessor",
)

# Load saved model schemas and threshold metadata
model_feature_schema = load_json(model_feature_schema_path)
feature_metadata = load_json(feature_schema_path)
threshold_summary = load_json(threshold_summary_path)

# Resolve final XGBoost model identity and family
FINAL_MODEL_NAME = parse_final_model_name(threshold_summary, feature_metadata)
FINAL_MODEL_FAMILY = "supervised"

# Resolve the XGBoost threshold used for live scoring
THRESHOLD_USED, THRESHOLD_SOURCE, FALLBACK_THRESHOLD_USED = resolve_xgboost_threshold(
    FINAL_MODEL_NAME,
    threshold_summary,
    threshold_summary_path,
)

# Define candidate paths for the selected XGBoost model
xgb_relatives = (
    ["models/xgboost_baseline.joblib", "xgboost_baseline.joblib"]
    if FINAL_MODEL_NAME == "XGB"
    else ["models/xgboost_tuned.joblib", "xgboost_tuned.joblib"]
)

# Resolve the selected XGBoost-family model path
xgb_model_path = resolve_first_existing(
    expand_candidate_paths(xgb_relatives),
    required=True,
    label="XGBoost risk model",
)

# Load the preprocessor and selected XGBoost-family model
preprocessor = joblib.load(preprocessor_path) if preprocessor_path and preprocessor_path.exists() else None
xgb_model = joblib.load(xgb_model_path)

# Load saved feature schema and model input metadata
SAVED_MODEL_FEATURES = list(model_feature_schema.get("model_features", feature_metadata.get("input_features", [])))
model_feature_names = feature_metadata.get("preprocessor_output", SAVED_MODEL_FEATURES)
MODEL_INPUT_FEATURE_NAMES = model_feature_names

# Load neutral defaults used when live features are unavailable
LIVE_NEUTRAL_DEFAULTS = model_feature_schema.get(
    "live_neutral_defaults",
    feature_metadata.get("live_neutral_defaults", {}),
)

# Define SHAP sampling and display limits
MAX_SHAP_GLOBAL_ROWS = 1_000
MAX_SHAP_DISPLAY_FEATURES = 20
MAX_TOP_SHAP_REVIEW_ROWS = 20
MAX_LOCAL_SHAP_FEATURES = 5

# Stop if the saved model feature schema is unavailable
if not SAVED_MODEL_FEATURES:
    raise ValueError("model_feature_schema.json is empty or missing the model_features list.")

# Warn when preprocessing artefacts are unavailable
if preprocessor is None:
    warnings.warn("preprocessor.joblib is missing.")

# Fall back to the default threshold if the loaded value is invalid
if not np.isfinite(THRESHOLD_USED):
    THRESHOLD_USED = DEFAULT_THRESHOLD

In [ ]:
# Define standard nameserver and domain status columns expected from WHOISDS exports
NS_COLUMNS = ["name_server_1", "name_server_2", "name_server_3", "name_server_4"]
STATUS_COLUMNS = ["domain_status_1", "domain_status_2", "domain_status_3", "domain_status_4"]

# Define alternative column names used to infer key WHOISDS fields
WHOISDS_COLUMN_CANDIDATES = {
    "domain": ["domain", "domain_name", "domainname", "host", "fqdn", "site", "url"],
    "registration_date": ["creation_date", "created", "created_date", "registration_date", "created on", "create_date"],
    "expiration_date": ["expiry_date", "expiration_date", "expires", "expire_date", "registry_expiry_date"],
    "registrar": ["registrar", "sponsoring_registrar", "registrar_name"],
    "status": ["status", "domain_status", "statuses"],
}

# Standardise column names for matching
def normalise_column_name(name: str) -> str:
    # Lowercase and replace non-alphanumeric runs with underscores
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")

# Infer the best matching column from a dataframe
def infer_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    # Build a lookup from normalised column names to original names
    normalized_map = {normalise_column_name(col): col for col in df.columns}
    for candidate in candidates:
        hit = normalized_map.get(normalise_column_name(candidate))
        if hit is not None:
            return hit
    # Fall back to partial normalised matches
    for col in df.columns:
        norm = normalise_column_name(col)
        if any(normalise_column_name(candidate) in norm for candidate in candidates):
            return col
    # Return None when no suitable column is found
    return None

# Sort WHOISDS input candidates by preferred source type and filename
def input_candidate_sort_key(path: Path) -> tuple[int, str]:
    # Prioritise zip archives before direct spreadsheet or CSV files
    suffix = path.suffix.lower()
    source_priority = 0 if suffix == ".zip" else 1
    # Return the sort key for deterministic input selection
    return (source_priority, path.name.lower())

# Sort extracted workbook candidates by preferred naming and file type
def extracted_workbook_sort_key(path: Path) -> tuple[int, int, str]:
    # Prefer files that look like full database workbooks
    stem = path.stem.lower()
    preferred_score = int("full" in stem and "database" in stem)
    # Prefer xlsx over older workbook formats
    xlsx_priority = int(path.suffix.lower() == ".xlsx")
    # Return the sort key for deterministic workbook selection
    return (-preferred_score, -xlsx_priority, str(path).lower())

# Extract a WHOISDS archive and return the selected workbook
def extract_whoisds_archive(path: Path, extraction_root: Path) -> Path:
    # Define the extraction directory from the archive name
    extraction_dir = extraction_root / path.stem
    # Recreate the extraction directory for a clean run
    if extraction_dir.exists():
        shutil.rmtree(extraction_dir)
    extraction_dir.mkdir(parents=True, exist_ok=True)
    # Extract archive contents into the run directory
    with zipfile.ZipFile(path) as archive:
        archive.extractall(extraction_dir)
    # Find supported workbook files inside the extracted archive
    workbook_candidates = sorted(
        (
            extracted_path
            for extracted_path in extraction_dir.rglob("*")
            if extracted_path.is_file() and extracted_path.suffix.lower() in {".xlsx", ".xls"}
        ),
        key=extracted_workbook_sort_key,
    )
    # Select the highest-priority workbook
    selected_workbook = workbook_candidates[0]
    # Report the archive extraction result
    print(f"Extracted WHOISDS archive: {path.name} -> {extraction_dir}")
    # Report workbook selection when multiple workbooks are found
    if len(workbook_candidates) > 1:
        print(f"Found {len(workbook_candidates)} extracted workbooks. Using {selected_workbook.name} by workbook priority.")
        # Return the selected workbook path
    return selected_workbook

# Detect the WHOISDS source file to process
def detect_whoisds_input(directory: Path) -> tuple[Path, Path, str]:
    # Collect supported WHOISDS files from the raw input directory
    candidates = sorted(
        (path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_WHOISDS_SOURCE_EXTENSIONS),
        key=input_candidate_sort_key,
    )
    # Report source selection when multiple files are available
    if len(candidates) > 1:
        print(f"Found {len(candidates)} WHOISDS sources. Using {candidates[0].name} by source priority and filename.")
    # Select the highest-priority source
    selected_source = candidates[0]
    # Extract zip sources before loading
    if selected_source.suffix.lower() == ".zip":
        extracted_workbook = extract_whoisds_archive(selected_source, WHOISDS_PROCESSED_DIR / "Extracted Databases")
        return extracted_workbook, selected_source, selected_source.stem
    # Return direct CSV or workbook sources unchanged
    return selected_source, selected_source, selected_source.stem

# Load a WHOISDS batch and report inferred field mappings
def load_whoisds_batch(path: Path) -> pd.DataFrame:
    # Detect input file type from suffix
    suffix = path.suffix.lower()
    # Load CSV or workbook source into a dataframe
    if suffix == ".csv":
        df = pd.read_csv(path, low_memory=False)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        df = pd.read_excel(path)
    # Infer key WHOISDS field columns
    inferred = {key: infer_column(df, candidates) for key, candidates in WHOISDS_COLUMN_CANDIDATES.items()}
    # Infer available nameserver and status columns
    inferred_nameservers = {key: infer_column(df, [key]) for key in NS_COLUMNS}
    inferred_statuses = {key: infer_column(df, [key]) for key in STATUS_COLUMNS}
    # Warn when expected fields are unavailable
    missing = [key for key, column in inferred.items() if column is None]
    if missing:
        warnings.warn("WHOISDS batch is missing some expected fields: " + ", ".join(missing))
    # Print dataset size and inferred mappings for traceability
    print(f"Loaded WHOISDS batch: {path.name}")
    print(f"Rows: {len(df):,} | Columns: {len(df.columns):,}")
    print("Inferred WHOISDS columns:")
    for key, column in inferred.items():
        print(f"  {key:18s} -> {column}")
    # Print inferred nameserver mappings
    print("Inferred WHOISDS nameserver columns:")
    for key, column in inferred_nameservers.items():
        print(f"  {key:18s} -> {column}")
    # Print inferred status mappings
    print("Inferred WHOISDS status columns:")
    for key, column in inferred_statuses.items():
        print(f"  {key:18s} -> {column}")
    # Return the loaded WHOISDS dataframe
    return df

# Detect the WHOISDS input source and run label
WHOISDS_INPUT_PATH, WHOISDS_SOURCE_PATH, WHOISDS_RUN_LABEL = detect_whoisds_input(WHOISDS_RAW_DIR)

# Create the run-specific output directory
WHOISDS_RUN_OUTPUT_DIR = WHOISDS_PROCESSED_DIR / WHOISDS_RUN_LABEL
WHOISDS_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create the run-specific debugging output directory
WHOISDS_DEBUGGING_OUTPUT_DIR = WHOISDS_RUN_OUTPUT_DIR / "Debugging"
WHOISDS_DEBUGGING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the raw WHOISDS batch
whoisds_raw = load_whoisds_batch(WHOISDS_INPUT_PATH)

# Build the inferred field map for core WHOISDS columns
WHOISDS_FIELD_MAP = {key: infer_column(whoisds_raw, candidates) for key, candidates in WHOISDS_COLUMN_CANDIDATES.items()}

# Capture available nameserver columns in source order without duplicates
WHOISDS_AVAILABLE_NAMESERVER_COLUMNS = list(dict.fromkeys(
    infer_column(whoisds_raw, [key]) for key in NS_COLUMNS if infer_column(whoisds_raw, [key]) is not None
))

# Capture available status columns in source order without duplicates
WHOISDS_AVAILABLE_STATUS_COLUMNS = list(dict.fromkeys(
    infer_column(whoisds_raw, [key]) for key in STATUS_COLUMNS if infer_column(whoisds_raw, [key]) is not None
))

Extracted WHOISDS archive: full-database-2026-06-08.zip -> /datasets/whoisds-processed/Extracted Databases/full-database-2026-06-08


## 3. Preprocessing

Live WHOISDS domains are normalised, validated, and deduplicated before filtering and enrichment.

In [ ]:
# Define the domain validation pattern used after normalisation
DOMAIN_REGEX = re.compile(
    r"^(?=.{1,253}$)(?!-)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}$",
    re.IGNORECASE,
)

# Define blacklist terms and field hints used for pre-filtering WHOISDS rows
BLACKLIST_KEYWORDS = ["wholesale", "whole.sale" , "sales.app", "sales.art", "lesales", "singles"]
BLACKLIST_FIELD_HINTS = ["domain", "fqdn", "hostname", "host", "site", "url"]

# Normalise domain-like text for blacklist searching
def normalise_searchable_domain_text(value: Any) -> str:
    # Convert missing values to empty text and lower-case non-missing values
    text = "" if value is None else str(value).strip().lower()
    # Remove common URL protocols
    text = text.replace("http://", "").replace("https://", "")
    # Remove query strings and fragments
    text = text.split("?", 1)[0].split("#", 1)[0]
    # Collapse repeated whitespace
    return re.sub(r"\s+", " ", text)

# Check whether text contains any blacklisted keyword
def contains_blacklisted_keyword(value: str, blacklist: list[str]) -> bool:
    value = str(value).lower()
    # Return True when any blacklist keyword is present
    return any(keyword.lower() in value for keyword in blacklist)

# Infer source columns that should be searched for blacklist keywords
def infer_blacklist_search_columns(df: pd.DataFrame, preferred_domain_column: Optional[str] = None) -> list[str]:
    # Collect columns whose names suggest domain-like content
    columns: list[str] = []
    # Add columns matching configured field hints
    for column in df.columns:
        normalised = normalise_column_name(column)
        if any(token in normalised for token in BLACKLIST_FIELD_HINTS):
            columns.append(column)
    # Prioritise the inferred domain column when available
    if preferred_domain_column and preferred_domain_column in df.columns and preferred_domain_column not in columns:
        columns.insert(0, preferred_domain_column)
    # Return unique columns while preserving order
    return list(dict.fromkeys(columns))

# Build combined searchable text for blacklist filtering
def build_blacklist_search_text(row: pd.Series, columns: list[str]) -> str:
    # Collect raw and normalised values from selected columns
    values: list[str] = []
    # Read each selected column from the row
    for column in columns:
        if column not in row.index:
            continue
        # Skip missing and empty values
        value = row.get(column)
        if pd.isna(value):
            continue
        raw_text = str(value).strip()
        if not raw_text:
            continue
        # Add raw, domain-normalised, and search-normalised variants
        values.extend([
            raw_text,
            normalise_domain(raw_text),
            normalise_searchable_domain_text(raw_text),
        ])
    # Return unique non-empty text fragments joined for keyword scanning
    return " | ".join(dict.fromkeys(value for value in values if value))

# Validate a normalised domain string against basic DNS naming rules
def is_valid_domain(domain: str) -> bool:
    # Reject missing, overlong, or whitespace-containing domains
    if not domain or len(domain) > 253 or " " in domain:
        return False
    # Require at least one dot
    if domain.count(".") < 1:
        return False
    # Reject repeated dots
    if ".." in domain:
        return False
    # Return whether the domain matches the validation regex
    return bool(DOMAIN_REGEX.match(domain))

# Extract the top-level domain from the registered domain
def extract_tld(domain: str) -> str:
    # Derive the registered domain first
    registered = get_registered_domain(domain)
    # Return empty text when no TLD can be derived
    if not registered or "." not in registered:
        return ""
    # Return the final label as the TLD
    parts = registered.split(".")
    return parts[-1]

# Resolve the source column containing domain values
domain_source_col = WHOISDS_FIELD_MAP.get("domain") or next(iter(whoisds_raw.columns), None)

# Copy raw WHOISDS rows before preprocessing
whoisds_work = whoisds_raw.copy()

# Normalise domain values into a standard domain column
whoisds_work["domain"] = whoisds_work[domain_source_col].map(normalise_domain)

# Infer columns to scan for blacklist keywords
blacklist_search_columns = infer_blacklist_search_columns(whoisds_work, domain_source_col)

# Ensure the normalised domain column is included in blacklist scanning
if "domain" not in blacklist_search_columns:
    blacklist_search_columns.append("domain")

# Build searchable blacklist text for each row
whoisds_work["blacklist_search_text"] = whoisds_work.apply(
    lambda row: build_blacklist_search_text(row, blacklist_search_columns),
    axis=1,
)

# Count keyword-level blacklist hits before filtering
BLACKLIST_HIT_COUNTS = {
    keyword: int(
        whoisds_work["blacklist_search_text"].map(
            lambda value, keyword=keyword: contains_blacklisted_keyword(value, [keyword])
        ).sum()
    )
    for keyword in BLACKLIST_KEYWORDS
}

# Flag rows containing any blacklisted keyword
whoisds_work["blacklist_keyword_flag"] = whoisds_work["blacklist_search_text"].map(
    lambda value: int(contains_blacklisted_keyword(value, BLACKLIST_KEYWORDS))
)

# Record row count before blacklist filtering
before_blacklist = len(whoisds_work)

# Remove blacklisted rows from the working dataset
whoisds_work = whoisds_work[whoisds_work["blacklist_keyword_flag"].eq(0)].copy()

# Count how many rows were removed by blacklist filtering
BLACKLIST_REMOVED_COUNT = before_blacklist - len(whoisds_work)

# Drop temporary blacklist helper columns
whoisds_work = whoisds_work.drop(columns=["blacklist_search_text", "blacklist_keyword_flag"], errors="ignore")

# Extract TLD values for remaining domains
whoisds_work["tld"] = whoisds_work["domain"].map(extract_tld)

# Capture the preprocessing timestamp
OBSERVED_AT = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)

# Flag valid normalised domains
whoisds_work["is_valid_domain"] = whoisds_work["domain"].fillna("").map(is_valid_domain)

# Keep only valid domains
whoisds_valid = whoisds_work[whoisds_work["is_valid_domain"]].copy()

# Remove duplicate domains before live enrichment and scoring
whoisds_valid = whoisds_valid.drop_duplicates(subset=["domain"]).reset_index(drop=True)

# Report blacklist and validation outcomes
print(f"Blacklist-removed rows     : {BLACKLIST_REMOVED_COUNT:,}")
print(f"Blacklist keyword hits     : {BLACKLIST_HIT_COUNTS}")
print(f"Valid unique WHOISDS domains: {len(whoisds_valid):,}")

## 4. Baseline Feature Engineering

Lexical, WHOIS, DNS, and TLS features are created from the live WHOISDS records and aligned with the saved baseline feature schema. This is done so that the live scoring workflow uses the same feature names, ordering, and preprocessing as the offline-trained model.

In [ ]:
# Define Maltese bank and finance brand indicators
BANK_FLAGS = [
    "bov",
    "bankofvalletta",
    "hsbc",
    "centralbank",
    "apsbank",
    "fcmbank",
    "medirect",
    "lombardmalta",
    "sparkasse",
    "fimbank",
    "ferratumbank",
    "izolabank",
    "bnf",
    "merkantibank",
    "iigbank",
    "novumbank",
    "credorax",
    "agribank",
    "akbank",
    "garantibbva",
    "nbg",
    "eccm",
]

# Define Maltese government and public-service indicators
MALTA_GOV_FLAGS = [
    "govmt",
    "mtgov",
    "identita",
    "servizzgov",
    "mita",
    "lesa",
]

# Define Malta context indicators
MALTA_FLAGS = [
    "malta",
    "maltese",
    "gozo",
]

# Define generic government, finance, account, and typosquatting indicators
GENERIC_GOV_FLAGS = ["gov", "government"]
FINANCE_FLAGS = ["payment", "invoice", "card", "wallet", "bank", "tax", "refund"]
ACCOUNT_FLAGS = [
    "login",
    "signin",
    "account",
    "verify",
    "secure",
    "update",
    "confirm",
    "reset",
    "auth",
    "support",
    "identity",
]
TYPOSQUAT_PATTERNS = ["b0v", "g0vmt", "mtg0v"]

# Calculate Shannon entropy for a string
def shannon_entropy(value: str) -> float:
    # Return zero entropy for empty values
    if not value:
        return 0.0
    # Count character frequencies
    counts = Counter(value)
    total = len(value)
    # Compute entropy from character frequency distribution
    return float(
        -sum((count / total) * math.log2(count / total) for count in counts.values())
    )

# Convert text to compact lowercase alphanumeric form
def compact_text(value: Any) -> str:
    # Return empty text for missing values
    if value is None:
        return ""
    # Remove non-alphanumeric characters for loose matching
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

# Extract the registered-domain stem before the final suffix
def domain_stem(domain: Any) -> str:
    # Derive the registered domain first
    registered = get_registered_domain(str(domain or ""))
    # Return empty text when no registered domain is available
    if not registered:
        return ""
    # Remove the final suffix label and lowercase the stem
    return registered.rsplit(".", 1)[0].lower()

# Split the registered-domain stem into alphanumeric tokens
def tokenise_domain(domain: Any) -> list[str]:
    # Return non-empty tokens from the domain stem
    return [token for token in re.split(r"[^a-z0-9]+", domain_stem(domain)) if token]

# Return unique tokens from the registered-domain stem
def domain_tokens(domain: Any) -> set[str]:
    # Convert token list into a set for keyword lookup
    return set(tokenise_domain(domain))

# Normalise domain text for boundary-aware keyword matching
def normalise_domain_text_for_matching(value: Any) -> str:
    # Lowercase and strip the incoming value
    text = str(value or "").strip().lower()
    # Keep domain separators while replacing other characters with spaces
    text = re.sub(r"[^a-z0-9._-]+", " ", text)
    # Convert dot, underscore, and hyphen separators into spaces
    text = re.sub(r"[._-]+", " ", text)
    # Collapse repeated whitespace
    return re.sub(r"\s+", " ", text).strip()

# Check whether a brand keyword safely matches a domain
def has_safe_brand_match(domain: Any, keyword: str) -> bool:
    # Prepare keyword, domain tokens, and compact domain stem
    keyword_lower = str(keyword).lower()
    tokens = domain_tokens(domain)
    compact = compact_text(domain_stem(domain))
    # Allow longer brand keywords to match tokens
    if len(keyword_lower) > 4:
        return keyword_lower in tokens or keyword_lower in compact
    # Require short brand keywords to match a full token
    return keyword_lower in tokens

# Check whether a domain matches a keyword group for a specific category
def has_keyword_group_match(value: Any, keywords: list[str], category: str) -> int:
    # Prepare normalised domain text, tokens, and compact text
    value_text = normalise_domain(value)
    tokens = domain_tokens(value_text)
    normalised_text = normalise_domain_text_for_matching(domain_stem(value_text))
    compact_value = compact_text(value_text)
    # Apply category-specific keyword matching rules
    for keyword in keywords:
        keyword_lower = str(keyword).lower()
        # Match brand terms
        if category == "brand":
            if has_safe_brand_match(value_text, keyword_lower):
                return 1
            continue
        # Match Maltese government terms using raw or compact text
        if category == "malta_government":
            compact_keyword = compact_text(keyword_lower)
            if keyword_lower in value_text or (compact_keyword and compact_keyword in compact_value):
                return 1
            continue
        # Match typosquatting patterns using compact text
        if category == "typosquat":
            if keyword_lower in compact_value:
                return 1
            continue
        # Match Malta context terms using tokens or separator boundaries
        if category == "malta_context":
            if keyword_lower in tokens:
                return 1
            if re.search(rf"(^|[ ._-]){re.escape(keyword_lower)}([ ._-]|$)", normalised_text):
                return 1
            continue
        # Match generic government, finance, and account terms as full tokens
        if category in {"generic_gov", "finance", "account"} and keyword_lower in tokens:
            return 1
    # Return zero when no keyword matches
    return 0

# Convert selected dataframe columns to integer flags
def ensure_int_columns(df: pd.DataFrame, columns: list[str]) -> None:
    # Coerce each available column to integer values
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0).astype(int)

# Count digits in a string
def count_digits(value: str) -> int:
    return sum(char.isdigit() for char in value)

# Count hyphens in a string
def count_hyphens(value: str) -> int:
    return value.count("-")

# Count subdomain labels before the registered domain
def count_subdomains(value: str) -> int:
    # Estimate subdomain count from the number of labels
    parts = value.split(".")
    return max(len(parts) - 2, 0)

# Count domain stem tokens
def token_count(value: str) -> int:
    # Return the number of alphanumeric domain tokens
    return len(tokenise_domain(value))

# Apply rule-based contextual and lexical features to WHOISDS domains
def apply_rule_feature_set(df: pd.DataFrame) -> pd.DataFrame:
    scored = df.copy()
    # Add registered domain and stem fields
    scored["registered_domain"] = scored["domain"].map(get_registered_domain)
    scored["domain_stem"] = scored["domain"].map(domain_stem)
    # Add keyword-based contextual flags
    scored["bank_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, BANK_FLAGS, "brand"))
    scored["malta_gov_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, MALTA_GOV_FLAGS, "malta_government"))
    scored["malta_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, MALTA_FLAGS, "malta_context"))
    scored["typosquatting_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, TYPOSQUAT_PATTERNS, "typosquat"))
    scored["generic_gov_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, GENERIC_GOV_FLAGS, "generic_gov"))
    scored["finance_keyword_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, FINANCE_FLAGS, "finance"))
    scored["account_keyword_flag"] = scored["domain"].map(lambda value: has_keyword_group_match(value, ACCOUNT_FLAGS, "account"))
    # Combine finance and account indicators into a suspicious keyword flag
    scored["suspicious_keyword_flag"] = (
        scored["finance_keyword_flag"].astype(int)
        | scored["account_keyword_flag"].astype(int)
    ).astype(int)
    # Combine Malta, Malta bank and finance brands and Maltese government and public-sector signals into a Malta relevance flag
    scored["malta_relevance_flag"] = (
        scored["malta_gov_flag"].astype(int)
        | scored["typosquatting_flag"].astype(int)
        | (
            scored["malta_flag"].astype(int)
            & (
                scored["bank_flag"].astype(int)
                | scored["generic_gov_flag"].astype(int)
                | scored["suspicious_keyword_flag"].astype(int)
            )
        )
    ).astype(int)
    # Add lexical and structural domain features
    scored["domain_entropy"] = scored["domain_stem"].map(shannon_entropy)
    scored["domain_length"] = scored["domain"].fillna("").str.len().astype(int)
    scored["digit_count"] = scored["domain"].fillna("").map(count_digits).astype(int)
    scored["hyphen_count"] = scored["domain"].fillna("").map(count_hyphens).astype(int)
    scored["subdomain_count"] = scored["domain"].fillna("").map(count_subdomains).astype(int)
    scored["shannon_entropy"] = scored["domain"].fillna("").map(shannon_entropy).astype(float)
    scored["tld_length"] = scored["tld"].fillna("").astype(str).str.len().astype(int)
    scored["token_count"] = scored["domain"].fillna("").map(token_count).astype(int)
    scored["has_digits"] = (scored["digit_count"] > 0).astype(int)
    scored["has_hyphen"] = (scored["hyphen_count"] > 0).astype(int)
    # Rule features are stored as integers
    ensure_int_columns(
        scored,
        [
            "bank_flag",
            "malta_gov_flag",
            "malta_flag",
            "typosquatting_flag",
            "generic_gov_flag",
            "finance_keyword_flag",
            "account_keyword_flag",
            "suspicious_keyword_flag",
            "malta_relevance_flag",
            "has_digits",
            "has_hyphen",
        ],
    )
    # Return the rule-feature dataframe
    return scored

# Apply rule-based feature engineering to valid WHOISDS domains
whoisds_valid = apply_rule_feature_set(whoisds_valid)

# Report valid and feature-ready WHOISDS row counts
print(f"Valid WHOISDS domains      : {len(whoisds_valid):,}")
print(f"Feature-ready WHOISDS rows : {len(whoisds_valid):,}")

In [ ]:
# Define numeric lexical feature columns expected for live scoring
NUMERIC_LEXICAL_COLUMNS = [
    "domain_length",
    "digit_count",
    "hyphen_count",
    "subdomain_count",
    "shannon_entropy",
    "tld_length",
    "token_count",
    "has_digits",
    "has_hyphen",
]

# Identify lexical feature columns available in the live dataframe
available_numeric_lexical_columns = [
    column for column in NUMERIC_LEXICAL_COLUMNS if column in whoisds_valid.columns
]

# Identify lexical feature columns missing from the live dataframe
missing_numeric_lexical_columns = [
    column for column in NUMERIC_LEXICAL_COLUMNS if column not in whoisds_valid.columns
]

# Coerce available lexical feature columns to numeric values
if available_numeric_lexical_columns:
    whoisds_valid[available_numeric_lexical_columns] = whoisds_valid[available_numeric_lexical_columns].apply(
        pd.to_numeric,
        errors="coerce",
    )

# Report how many lexical features were prepared
print(f"Prepared lexical features: {len(available_numeric_lexical_columns):,}")

# Report any skipped lexical features
if missing_numeric_lexical_columns:
    print("Skipped missing lexical columns:", ", ".join(missing_numeric_lexical_columns))

In [ ]:
# Parse a string or collection into a clean list of text values
def parse_collection_like(value: Any) -> list[str]:
    # Return an empty list for missing values
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    # Convert existing collections into stripped strings
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    # Convert scalar values into stripped text
    text = str(value).strip()
    # Return empty list for empty text
    if not text:
        return []
    # Remove surrounding square brackets from list-like strings
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1]
    # Split semicolon, comma, or pipe separated values
    parts = re.split(r"[;,|]", text)
    # Return cleaned non-empty parts
    return [part.strip().strip("'").strip('"') for part in parts if part.strip()]

# Add standardised nameserver fields and nameserver count
def add_nameserver_features(df: pd.DataFrame, source_columns: Optional[list[str]] = None) -> pd.DataFrame:
    # Work on a copy to preserve the input dataframe
    df = df.copy()
    # Keep available source nameserver columns
    available_ns_cols = [c for c in (source_columns or NS_COLUMNS) if c in df.columns]
    # Copy available nameserver values into standard output columns
    for target_col, source_col in zip(NS_COLUMNS, available_ns_cols):
        df[target_col] = df[source_col].fillna("").astype(str).str.strip().str.lower()
    # Fill missing standard nameserver columns with empty strings
    for target_col in NS_COLUMNS[len(available_ns_cols):]:
        df[target_col] = ""
    # Build a cleaned nameserver frame
    ns_frame = df[NS_COLUMNS].fillna("").astype(str).apply(lambda col: col.str.strip().str.lower())
    # Count populated nameserver values
    df["nameserver_count"] = ns_frame.ne("").sum(axis=1).astype(int)
    # Return dataframe with nameserver features
    return df

# Add standardised domain status fields and status count
def add_status_features(df: pd.DataFrame, source_columns: Optional[list[str]] = None, fallback_column: Optional[str] = None) -> pd.DataFrame:
    # Work on a copy to preserve the input dataframe
    df = df.copy()
    # Keep available source status columns
    available_status_cols = [c for c in (source_columns or STATUS_COLUMNS) if c in df.columns]
    # Copy available status values into standard output columns
    for target_col, source_col in zip(STATUS_COLUMNS, available_status_cols):
        df[target_col] = df[source_col].fillna("").astype(str).str.strip().str.lower()
    # Fill missing standard status columns with empty strings
    for target_col in STATUS_COLUMNS[len(available_status_cols):]:
        df[target_col] = ""
    # Use fallback collection-like status column when separate status columns are unavailable
    if not available_status_cols and fallback_column and fallback_column in df.columns:
        status_values = df[fallback_column].map(parse_collection_like).map(lambda values: [str(value).strip().lower() for value in values if str(value).strip()])
        # Spread parsed fallback status values across standard status columns
        for idx, target_col in enumerate(STATUS_COLUMNS):
            df[target_col] = status_values.map(lambda values, idx=idx: values[idx] if idx < len(values) else "")
    # Build a cleaned status frame
    status_frame = df[STATUS_COLUMNS].fillna("").astype(str).apply(lambda col: col.str.strip().str.lower())
    # Count populated status values
    df["status_count"] = status_frame.ne("").sum(axis=1).astype(int)
    # Return dataframe with status features
    return df

# Infer whether registrar or domain text suggests privacy protection or redaction
def infer_privacy_flag(row: pd.Series) -> int:
    # Select registrar and domain text candidates
    candidate_cols = [WHOISDS_FIELD_MAP.get("registrar"), "domain"]
    # Collect available candidate text chunks
    text_chunks = []
    for column in candidate_cols:
        if column and column in row.index and pd.notna(row[column]):
            text_chunks.append(str(row[column]).lower())
    # Combine text for privacy keyword detection
    combined = " ".join(text_chunks)
    # Return one when privacy or redaction terms are found
    return int(any(term in combined for term in ["privacy", "redacted", "redaction", "whoisguard", "proxy"]))

# Parse mapped registration and expiration date columns
for mapped_name in ["registration_date", "expiration_date"]:
    # Resolve the original source column for the mapped date field
    source_col = WHOISDS_FIELD_MAP.get(mapped_name)
    # Parse source dates when available
    if source_col:
        parsed_series = whoisds_valid[source_col].map(parse_datetime_safe)
        whoisds_valid[mapped_name] = pd.to_datetime(parsed_series, errors="coerce")
    # Fill missing mapped dates with missing timestamps
    else:
        whoisds_valid[mapped_name] = pd.NaT

# Add the live observation timestamp
whoisds_valid["observed_at"] = OBSERVED_AT

# Compute domain lifecycle features from observed and WHOIS dates
whoisds_valid = compute_domain_lifecycle_features(
    whoisds_valid,
    observed_col="observed_at",
    registration_col="registration_date",
    expiration_col="expiration_date",
)

# Add nameserver count and standardised nameserver columns
whoisds_valid = add_nameserver_features(whoisds_valid, WHOISDS_AVAILABLE_NAMESERVER_COLUMNS)

# Add status count and standardised status columns
whoisds_valid = add_status_features(whoisds_valid, WHOISDS_AVAILABLE_STATUS_COLUMNS, WHOISDS_FIELD_MAP.get("status"))

# Resolve the registrar source column
registrar_col = WHOISDS_FIELD_MAP.get("registrar")

# Standardise registrar text
whoisds_valid["registrar"] = (whoisds_valid[registrar_col].fillna("").astype(str).str.strip() if registrar_col else "")
whoisds_valid["registrar_present_flag"] = whoisds_valid["registrar"].ne("").astype(int)

# Add registrar presence and privacy or redaction flags
whoisds_valid["privacy_or_redaction_flag"] = whoisds_valid.apply(infer_privacy_flag, axis=1).astype(int)

# Report parsed WHOIS feature availability
print(f"Registration dates parsed : {int(whoisds_valid['registration_date'].notna().sum()):,}")
print(f"Nameserver values present : {int(whoisds_valid['nameserver_count'].gt(0).sum()):,}")
print(f"Registrar values present  : {int(whoisds_valid['registrar_present_flag'].sum()):,}")

## 5. Domain Context / Crawler Feature Engineering

Contextual indicators are created from the live WHOISDS domains to identify records that are Malta-related (Malta / Maltese banking or finance / Maltese Government / Public-service) or generic government / finance / account indicators.

In [ ]:
# Initialise caches for network enrichment results
DNS_CACHE: dict[str, dict[str, Any]] = {}
TLS_CACHE: dict[str, dict[str, Any]] = {}
HTTP_CACHE: dict[str, dict[str, Any]] = {}

# Create a reusable HTTP session with the configured user agent and redirect limit
HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update({"User-Agent": USER_AGENT})
HTTP_SESSION.max_redirects = MAX_HTTP_REDIRECTS

# Define keywords used to identify parked-domain pages
PARKED_PAGE_KEYWORDS = [
    "domain for sale",
    "buy this domain",
    "this domain is parked",
    "sedo",
    "dan.com",
    "parkingcrew",
    "bodis",
    "namecheap parking",
    "godaddy",
]

# Define keywords used to identify phishing-like page content
PHISHING_PAGE_KEYWORDS = [
    "login",
    "verify account",
    "sign in",
    "password",
    "mfa",
    "security check",
    "banking",
    "authenticate",
]

# Define error text fragments that indicate DNS resolution failures
DNS_ERROR_KEYWORDS = [
    "name or service not known",
    "temporary failure in name resolution",
    "nodename nor servname",
    "getaddrinfo failed",
    "no address associated with hostname",
    "failed to resolve",
]

# Parse certificate timestamp strings into pandas timestamps
def _parse_cert_time(value: Optional[str]) -> pd.Timestamp:
    # Return missing timestamp for empty values
    if not value:
        return pd.NaT
    # Try common certificate time formats
    for fmt in ["%b %d %H:%M:%S %Y %Z", "%Y%m%d%H%M%SZ"]:
        try:
            dt = datetime.strptime(value, fmt).replace(tzinfo=timezone.utc)
            return pd.Timestamp(dt)
        except Exception:
            continue
    # Return missing timestamp when parsing fails
    return pd.NaT

# Normalise page text for keyword matching
def _normalise_page_text(value: Any) -> str:
    # Collapse whitespace and lowercase the text
    return re.sub(r"\s+", " ", str(value or "")).strip().lower()

# Truncate text to a fixed maximum length
def _truncate_text(value: Any, max_chars: int = 250) -> str:
    # Collapse whitespace before truncating
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    # Return text unchanged when it is within the limit
    if len(text) <= max_chars:
        return text
    # Truncate long text and add an ellipsis
    return text[: max_chars - 3].rstrip() + "..."

# Check whether normalised text contains any keyword
def _contains_keyword(text: str, keywords: list[str]) -> int:
    # Normalise text before keyword scanning
    normalised = _normalise_page_text(text)
    # Return one when any keyword is found
    return int(any(keyword in normalised for keyword in keywords))

# Normalise request exception messages
def _request_error_message(exc: Exception) -> str:
    # Convert exception text into a normalised message
    return _normalise_page_text(str(exc))

# Detect DNS-related errors from request messages
def _dns_error_from_message(message: str) -> bool:
    # Match known DNS error fragments
    return any(hint in message for hint in DNS_ERROR_KEYWORDS)

# Enrich a domain with DNS record counts and lookup status
def enrich_dns(domain: str) -> dict[str, Any]:
    # Return cached DNS enrichment when available
    if domain in DNS_CACHE:
        return DNS_CACHE[domain]
    # Configure a DNS resolver with notebook timeouts
    resolver = dns.resolver.Resolver()
    resolver.timeout = DNS_TIMEOUT
    resolver.lifetime = DNS_TIMEOUT
    # Initialise DNS result fields
    result = {
        "domain": domain,
        "dns_a_count": 0,
        "dns_aaaa_count": 0,
        "dns_mx_count": 0,
        "dns_ns_count": 0,
        "dns_txt_count": 0,
        "dns_soa_count": 0,
        "dns_has_a": 0,
        "dns_has_mx": 0,
        "dns_has_soa": 0,
        "dns_lookup_success": 0,
        "dns_error_type": None,
    }
    # Track successful record categories
    success_count = 0
    # Map DNS record types to output count columns
    mapping = {
        "A": "dns_a_count",
        "AAAA": "dns_aaaa_count",
        "MX": "dns_mx_count",
        "NS": "dns_ns_count",
        "TXT": "dns_txt_count",
        "SOA": "dns_soa_count",
    }
    # Query each DNS record type
    for record_type, count_key in mapping.items():
        try:
            answers = resolver.resolve(domain, record_type, raise_on_no_answer=False)
            if answers.rrset is not None:
                count = len(answers)
                result[count_key] = int(count)
                success_count += int(count > 0)
        # Store the first expected DNS error type
        except (dns.resolver.NXDOMAIN, dns.resolver.NoNameservers, dns.resolver.NoAnswer, dns.exception.Timeout) as exc:
            if result["dns_error_type"] is None:
                result["dns_error_type"] = type(exc).__name__
        # Store the first unexpected DNS error type
        except Exception as exc:
            if result["dns_error_type"] is None:
                result["dns_error_type"] = f"unexpected_{type(exc).__name__}"
    # Derive DNS presence and success flags
    result["dns_has_a"] = int(result["dns_a_count"] > 0)
    result["dns_has_mx"] = int(result["dns_mx_count"] > 0)
    result["dns_has_soa"] = int(result["dns_soa_count"] > 0)
    result["dns_lookup_success"] = int(success_count > 0)
    # Cache and return the DNS enrichment result
    DNS_CACHE[domain] = result
    return result

# Enrich a domain with TLS certificate details
def enrich_tls(domain: str) -> dict[str, Any]:
    # Return cached TLS enrichment when available
    if domain in TLS_CACHE:
        return TLS_CACHE[domain]
    # Initialise TLS result fields
    result = {
        "domain": domain,
        "tls_handshake_success": 0,
        "tls_issuer_common_name": None,
        "tls_issuer_organization": None,
        "tls_subject_common_name": None,
        "tls_not_before": pd.NaT,
        "tls_not_after": pd.NaT,
        "tls_validity_days": np.nan,
        "tls_days_until_expiry": np.nan,
        "tls_self_signed_flag": np.nan,
        "tls_error_type": None,
    }
    # Create a permissive TLS context for metadata collection
    context = ssl.create_default_context()
    context.check_hostname = False
    context.verify_mode = ssl.CERT_NONE
    # Attempt TLS connection and certificate extraction
    try:
        with socket.create_connection((domain, TLS_PORT), timeout=TLS_TIMEOUT) as sock:
            with context.wrap_socket(sock, server_hostname=domain) as ssock:
                cert = ssock.getpeercert()
        # Mark successful TLS handshake
        result["tls_handshake_success"] = 1
        # Extract issuer and subject certificate fields
        issuer_pairs = [item for rdn in cert.get("issuer", []) for item in rdn]
        subject_pairs = [item for rdn in cert.get("subject", []) for item in rdn]
        issuer_map = {key: value for key, value in issuer_pairs}
        subject_map = {key: value for key, value in subject_pairs}
        result["tls_issuer_common_name"] = issuer_map.get("commonName")
        result["tls_issuer_organization"] = issuer_map.get("organizationName")
        result["tls_subject_common_name"] = subject_map.get("commonName")
        # Parse certificate validity dates
        not_before = _parse_cert_time(cert.get("notBefore"))
        not_after = _parse_cert_time(cert.get("notAfter"))
        result["tls_not_before"] = not_before
        result["tls_not_after"] = not_after
        # Calculate validity duration and days until expiry
        if pd.notna(not_before) and pd.notna(not_after):
            result["tls_validity_days"] = int((not_after - not_before).days)
            result["tls_days_until_expiry"] = int((not_after - pd.Timestamp.now(tz="UTC")).days)
        # Infer self-signed certificates from matching issuer and subject common names
        issuer_cn = str(result["tls_issuer_common_name"] or "").strip().lower()
        subject_cn = str(result["tls_subject_common_name"] or "").strip().lower()
        result["tls_self_signed_flag"] = int(bool(issuer_cn) and issuer_cn == subject_cn)
    # Store TLS error categories
    except ssl.SSLError as exc:
        result["tls_error_type"] = f"ssl_{type(exc).__name__}"
    except (socket.timeout, TimeoutError) as exc:
        result["tls_error_type"] = f"timeout_{type(exc).__name__}"
    except OSError as exc:
        result["tls_error_type"] = f"os_{type(exc).__name__}"
    except Exception as exc:
        result["tls_error_type"] = f"unexpected_{type(exc).__name__}"
    # Cache and return the TLS enrichment result
    TLS_CACHE[domain] = result
    return result

# Extract meta description text from parsed HTML
def _extract_meta_description(soup: BeautifulSoup) -> str:
    # Check standard description and Open Graph description tags
    meta_candidates = [
        soup.find("meta", attrs={"name": re.compile(r"^description$", re.IGNORECASE)}),
        soup.find("meta", attrs={"property": re.compile(r"^og:description$", re.IGNORECASE)}),
    ]
    # Return the first populated meta description
    for meta_tag in meta_candidates:
        if meta_tag is None:
            continue
        content = meta_tag.get("content")
        if content:
            return _truncate_text(unescape(content))
    # Return empty text when no description is available
    return ""

# Extract visible page body text from parsed HTML
def _extract_visible_body_text(soup: BeautifulSoup) -> str:
    # Copy parsed HTML before removing non-visible tags
    soup_copy = BeautifulSoup(str(soup), "html.parser")
    # Remove script, style, and noscript content
    for tag in soup_copy(["script", "style", "noscript"]):
        tag.decompose()
    # Extract body text when present or all text otherwise
    body_text = soup_copy.body.get_text(" ", strip=True) if soup_copy.body else soup_copy.get_text(" ", strip=True)
    # Decode entities and collapse whitespace
    body_text = re.sub(r"\s+", " ", unescape(body_text)).strip()
    # Return visible text
    return body_text

# Build a short summary of detected form and input elements
def _build_detected_form_summary(form_count: int, password_count: int, input_count: int) -> str:
    # Summarise login-like forms when password inputs are present
    if password_count > 0 and form_count > 0:
        return _truncate_text(
            f"Login form with {password_count} password field{'s' if password_count != 1 else ''}"
            + (f" and {input_count} input{'s' if input_count != 1 else ''}" if input_count > 0 else "")
        )
    # Summarise password inputs without forms
    if password_count > 0:
        return _truncate_text(f"{password_count} password field{'s' if password_count != 1 else ''} detected")
    # Summarise form and input counts
    if form_count > 0:
        summary = f"{form_count} form{'s' if form_count != 1 else ''} detected"
        if input_count > 0:
            summary += f"; {input_count} input{'s' if input_count != 1 else ''}"
        return _truncate_text(summary)
    # Summarise inputs without forms
    if input_count > 0:
        return _truncate_text(f"{input_count} input{'s' if input_count != 1 else ''} detected; no forms")
    # Return default summary when no forms are found
    return "No forms detected"

# Count external resources in HTML for a given tag and attribute
def _extract_external_resources(html: str, base_netloc: str, attr: str, tag: str) -> int:
    # Build a simple regex for the requested tag attribute
    pattern = rf"<{tag}[^>]+{attr}=[\"']?([^\"'>\s]+)"
    count = 0
    # Count absolute HTTP resources pointing to other hosts
    for target in re.findall(pattern, html, flags=re.IGNORECASE):
        parsed = urlparse(unescape(target))
        if parsed.scheme in {"http", "https"} and parsed.netloc and parsed.netloc.lower() != base_netloc.lower():
            count += 1
    # Return external resource count
    return count

# Enrich a domain with HTTP crawl and page content signals
def enrich_http(domain: str) -> dict[str, Any]:
    # Return cached HTTP enrichment when available
    if domain in HTTP_CACHE:
        return HTTP_CACHE[domain]
    # Initialise HTTP result fields
    result = {
        "domain": domain,
        "http_success": 0,
        "final_url": None,
        "http_status": np.nan,
        "redirect_count": 0,
        "uses_https": 0,
        "page_title": "",
        "page_meta_description": "",
        "visible_text_snippet": "",
        "detected_form_summary": "",
        "content_detected": 0,
        "parked_domain_flag": 0,
        "phishing_keyword_flag": 0,
        "html_title_length": 0,
        "html_form_count": 0,
        "html_input_count": 0,
        "html_password_input_count": 0,
        "html_external_link_count": 0,
        "html_external_script_count": 0,
        "html_iframe_count": 0,
        "page_text_length": 0,
        "crawl_status": "connection_error",
        "timeout_flag": 0,
        "dns_error_flag": 0,
        "ssl_error_flag": 0,
    }
    # Try HTTPS first and then HTTP
    for scheme in ["https", "http"]:
        url = f"{scheme}://{domain}"
        # Attempt to retrieve and parse the page
        try:
            response = HTTP_SESSION.get(url, timeout=HTTP_TIMEOUT, allow_redirects=True)
            html = response.text or ""
            parsed_final = urlparse(str(response.url))
            soup = BeautifulSoup(html, "html.parser")
            # Extract page title, meta description, and visible text
            title_text = _truncate_text(soup.title.get_text(" ", strip=True) if soup.title else "")
            page_meta_description = _extract_meta_description(soup)
            body_text = _extract_visible_body_text(soup)
            visible_text_snippet = _truncate_text(body_text)
            # Build combined text for parked and phishing keyword detection
            signal_text = " ".join(
                part for part in [title_text, page_meta_description, body_text, str(response.url), parsed_final.netloc] if part
            )
            # Count forms, inputs, password inputs, and page content indicators
            html_form_count = len(soup.find_all("form"))
            html_input_count = len(soup.find_all("input"))
            html_password_input_count = len(
                soup.find_all("input", attrs={"type": re.compile(r"password", re.IGNORECASE)})
            )
            detected_form_summary = _build_detected_form_summary(
                html_form_count,
                html_password_input_count,
                html_input_count,
            )
            content_detected = int(
                bool(title_text)
                or len(body_text) >= 40
                or html_form_count > 0
                or html_input_count > 0
            )
            # Store HTTP, content, and page structure signals
            result.update({
                "http_success": 1,
                "final_url": str(response.url),
                "http_status": int(response.status_code),
                "redirect_count": max(len(response.history), 0),
                "uses_https": int(parsed_final.scheme.lower() == "https"),
                "page_title": title_text,
                "page_meta_description": page_meta_description,
                "visible_text_snippet": visible_text_snippet,
                "detected_form_summary": detected_form_summary,
                "content_detected": content_detected,
                "parked_domain_flag": _contains_keyword(signal_text, PARKED_PAGE_KEYWORDS),
                "phishing_keyword_flag": _contains_keyword(signal_text, PHISHING_PAGE_KEYWORDS),
                "html_title_length": len(title_text),
                "html_form_count": html_form_count,
                "html_input_count": html_input_count,
                "html_password_input_count": html_password_input_count,
                "html_external_link_count": _extract_external_resources(html, parsed_final.netloc, "href", "a"),
                "html_external_script_count": _extract_external_resources(html, parsed_final.netloc, "src", "script"),
                "html_iframe_count": len(soup.find_all("iframe")),
                "page_text_length": len(body_text),
            })
            # Derive crawl status from the populated result row
            result["crawl_status"] = normalise_site_status_from_row(pd.Series(result))
            # Cache and return the first successful crawl result
            HTTP_CACHE[domain] = result
            return result
        # Track timeout, SSL, redirect, DNS, and HTTP request failures
        except requests.Timeout:
            result["timeout_flag"] = 1
        except requests.exceptions.SSLError:
            result["ssl_error_flag"] = 1
        except requests.TooManyRedirects:
            pass
        except requests.exceptions.ConnectionError as exc:
            message = _request_error_message(exc)
            if _dns_error_from_message(message):
                result["dns_error_flag"] = 1
        except requests.RequestException as exc:
            response = getattr(exc, "response", None)
            if response is not None and response.status_code is not None:
                result["http_status"] = int(response.status_code)
        except Exception:
            pass
    # Derive crawl status after all schemes fail
    result["crawl_status"] = normalise_site_status_from_row(pd.Series(result))
    # Cache and return the failed crawl result
    HTTP_CACHE[domain] = result
    return result

# Apply an enrichment function to domains with progress reporting
def enrich_with_progress(domains: list[str], enrich_func: Any, label: str) -> pd.DataFrame:
    # Return an empty dataframe when no domains are supplied
    if not domains:
        return pd.DataFrame()
    # Collect enrichment rows and total count
    rows = []
    total = len(domains)
    # Enrich each domain and report periodic progress
    for index, domain in enumerate(domains, start=1):
        rows.append(enrich_func(domain))
        if index == 1 or index == total or index % 10 == 0:
            progress_pct = (index / total) * 100.0
            print(f"{label}: {index:,}/{total:,} ({progress_pct:5.1f}%)")
    # Return enrichment results as a dataframe
    return pd.DataFrame(rows)

In [ ]:
# Start the live feature table from the valid WHOISDS domains
whoisds_features = whoisds_valid.copy()

# Define DNS enrichment columns expected in the live evidence table
DNS_ENRICHMENT_COLUMNS = [
    "domain", "dns_a_count", "dns_aaaa_count", "dns_mx_count", "dns_ns_count", "dns_txt_count", "dns_soa_count",
    "dns_has_a", "dns_has_mx", "dns_has_soa", "dns_lookup_success", "dns_error_type",
]

# Define TLS enrichment columns expected in the live evidence table
TLS_ENRICHMENT_COLUMNS = [
    "domain", "tls_handshake_success", "tls_issuer_common_name", "tls_issuer_organization", "tls_subject_common_name",
    "tls_not_before", "tls_not_after", "tls_validity_days", "tls_days_until_expiry", "tls_self_signed_flag", "tls_error_type",
]

# Define HTTP enrichment columns expected in the live evidence table
HTTP_ENRICHMENT_COLUMNS = [
    "domain", "http_success", "final_url", "http_status", "redirect_count", "uses_https", "page_title",
    "page_meta_description", "visible_text_snippet", "detected_form_summary",
    "content_detected", "parked_domain_flag", "phishing_keyword_flag", "timeout_flag", "dns_error_flag", "ssl_error_flag",
    "html_title_length", "html_form_count", "html_input_count", "html_password_input_count", "html_external_link_count",
    "html_external_script_count", "html_iframe_count", "page_text_length", "crawl_status",
]

# Define the full saved enrichment schema
SAVED_ENRICHMENT_COLUMNS = sorted(set(
    DNS_ENRICHMENT_COLUMNS
    + TLS_ENRICHMENT_COLUMNS
    + HTTP_ENRICHMENT_COLUMNS
    + ["enrichment_attempted"]
))

# Create an empty enrichment dataframe with the expected columns
def empty_enrichment_df(columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame(columns=columns)

# Create empty live enrichment frames when active enrichment is not run
dns_enrichment_live_df = empty_enrichment_df(DNS_ENRICHMENT_COLUMNS)
tls_enrichment_live_df = empty_enrichment_df(TLS_ENRICHMENT_COLUMNS)
http_enrichment_live_df = empty_enrichment_df(HTTP_ENRICHMENT_COLUMNS)

# Merge empty or populated DNS, TLS, and HTTP enrichment frames into the feature table
whoisds_enriched = whoisds_features.merge(dns_enrichment_live_df, on="domain", how="left")
whoisds_enriched = whoisds_enriched.merge(tls_enrichment_live_df, on="domain", how="left")
whoisds_enriched = whoisds_enriched.merge(http_enrichment_live_df, on="domain", how="left")

# Mark rows as not actively enriched in this configuration
whoisds_enriched["enrichment_attempted"] = 0

# Define default values for enrichment-derived fields
DEFAULT_ENRICHMENT_VALUES = {
    "dns_a_count": 0, "dns_aaaa_count": 0, "dns_mx_count": 0, "dns_ns_count": 0, "dns_txt_count": 0, "dns_soa_count": 0,
    "dns_has_a": 0, "dns_has_mx": 0, "dns_has_soa": 0, "dns_lookup_success": 0,
    "tls_handshake_success": 0, "tls_validity_days": np.nan, "tls_days_until_expiry": np.nan, "tls_self_signed_flag": np.nan,
    "http_success": 0, "http_status": np.nan, "redirect_count": 0, "uses_https": 0, "page_title": "", "page_meta_description": "", "visible_text_snippet": "", "detected_form_summary": "", "content_detected": 0,
    "parked_domain_flag": 0, "phishing_keyword_flag": 0, "timeout_flag": 0, "dns_error_flag": 0, "ssl_error_flag": 0,
    "html_title_length": 0, "html_form_count": 0, "html_input_count": 0, "html_password_input_count": 0,
    "html_external_link_count": 0, "html_external_script_count": 0, "html_iframe_count": 0, "page_text_length": 0,
    "crawl_status": "", "final_url": "",
}

# Fill missing enrichment fields with neutral defaults
for column, default_value in DEFAULT_ENRICHMENT_VALUES.items():
    if column in whoisds_enriched.columns:
        whoisds_enriched[column] = whoisds_enriched[column].fillna(default_value)

# Prepare DNS and TLS error text for derived error flags
dns_error_text = whoisds_enriched.get("dns_error_type", pd.Series("", index=whoisds_enriched.index)).fillna("").astype(str)
tls_error_text = whoisds_enriched.get("tls_error_type", pd.Series("", index=whoisds_enriched.index)).fillna("").astype(str)

# Derive a DNS error flag from explicit flags, lookup failures, and DNS error text
whoisds_enriched["dns_error_flag"] = (
    pd.to_numeric(whoisds_enriched.get("dns_error_flag", 0), errors="coerce").fillna(0).astype(int).eq(1)
    | (
        whoisds_enriched["dns_lookup_success"].fillna(0).astype(int).eq(0)
        & dns_error_text.str.contains(r"NXDOMAIN|NoNameservers|NoAnswer", regex=True, na=False)
    )
).astype(int)

# Derive an SSL error flag from explicit flags and TLS error text
whoisds_enriched["ssl_error_flag"] = (
    pd.to_numeric(whoisds_enriched.get("ssl_error_flag", 0), errors="coerce").fillna(0).astype(int).eq(1)
    | tls_error_text.str.contains(r"^ssl_", regex=True, na=False)
).astype(int)

# Derive a timeout flag from explicit flags and DNS or TLS timeout text
whoisds_enriched["timeout_flag"] = (
    pd.to_numeric(whoisds_enriched.get("timeout_flag", 0), errors="coerce").fillna(0).astype(int).eq(1)
    | dns_error_text.str.contains("Timeout", case=False, regex=False, na=False)
    | tls_error_text.str.contains("timeout", case=False, regex=False, na=False)
).astype(int)

# Set crawl status to unknown when enrichment was not attempted
whoisds_enriched["crawl_status"] = whoisds_enriched.apply(
    lambda row: "unknown" if int(row.get("enrichment_attempted", 0)) == 0 else normalise_site_status_from_row(row),
    axis=1,
)

# Report available DNS, TLS, and HTTP evidence counts
print(f"DNS evidence rows       : {int(whoisds_enriched['dns_lookup_success'].fillna(0).astype(int).sum()):,}")
print(f"TLS evidence rows       : {int(whoisds_enriched['tls_handshake_success'].fillna(0).astype(int).sum()):,}")
print(f"HTTP evidence rows      : {int(whoisds_enriched['http_success'].fillna(0).astype(int).sum()):,}")

## 6. Domain Crawler

Selected live WHOISDS domains are queried over HTTP and HTTPS to test whether a website is reachable, record the final resolved URL after redirects, capture the HTTP status code, extract the page title and visible text, and flag page-level indicators such as login terms, password fields, and HTML forms.

In [ ]:
# Enrich only prioritised high and medium matched domains
def selective_enrich_prioritised_domains(df: pd.DataFrame) -> pd.DataFrame:
    # Work on a copy to preserve the input dataframe
    prioritised = df.copy()
    # Select rows eligible for live DNS, TLS, and HTTP enrichment
    crawl_candidates = prioritised[
        matched_high_medium_mask(prioritised)
    ].copy()
    # Build a unique list of domains to enrich
    live_domains = crawl_candidates["domain"].drop_duplicates().tolist()
    # Run live enrichment when prioritised domains are available
    if live_domains:
        dns_enrichment_df = enrich_with_progress(live_domains, enrich_dns, "DNS enrichment")
        tls_enrichment_df = enrich_with_progress(live_domains, enrich_tls, "TLS enrichment")
        http_enrichment_df = enrich_with_progress(live_domains, enrich_http, "HTTP enrichment")
    # Use empty enrichment frames when no domains are selected
    else:
        dns_enrichment_df = empty_enrichment_df(DNS_ENRICHMENT_COLUMNS)
        tls_enrichment_df = empty_enrichment_df(TLS_ENRICHMENT_COLUMNS)
        http_enrichment_df = empty_enrichment_df(HTTP_ENRICHMENT_COLUMNS)
    # Remove stale enrichment columns before merging fresh enrichment evidence
    enrichment_base = prioritised.drop(columns=[column for column in SAVED_ENRICHMENT_COLUMNS if column != "domain" and column in prioritised.columns], errors="ignore")
    # Merge DNS, TLS, and HTTP enrichment evidence by domain
    enriched_df = enrichment_base.merge(dns_enrichment_df, on="domain", how="left")
    enriched_df = enriched_df.merge(tls_enrichment_df, on="domain", how="left")
    enriched_df = enriched_df.merge(http_enrichment_df, on="domain", how="left")
    # Mark domains that were selected for live enrichment
    enriched_df["enrichment_attempted"] = enriched_df["domain"].isin(set(live_domains)).astype(int)
    # Fill missing enrichment values with neutral defaults
    for column, default_value in DEFAULT_ENRICHMENT_VALUES.items():
        if column in enriched_df.columns:
            enriched_df[column] = enriched_df[column].fillna(default_value)
    # Prepare DNS and TLS error text for derived error flags
    dns_error_text = enriched_df.get("dns_error_type", pd.Series("", index=enriched_df.index)).fillna("").astype(str)
    tls_error_text = enriched_df.get("tls_error_type", pd.Series("", index=enriched_df.index)).fillna("").astype(str)
    # Derive DNS error flag from explicit flags, lookup failures, and DNS error text
    enriched_df["dns_error_flag"] = (
        pd.to_numeric(enriched_df.get("dns_error_flag", 0), errors="coerce").fillna(0).astype(int).eq(1)
        | (
            enriched_df["dns_lookup_success"].fillna(0).astype(int).eq(0)
            & dns_error_text.str.contains(r"NXDOMAIN|NoNameservers|NoAnswer", regex=True, na=False)
        )
    ).astype(int)
    # Derive SSL error flag from explicit flags and TLS error text
    enriched_df["ssl_error_flag"] = (
        pd.to_numeric(enriched_df.get("ssl_error_flag", 0), errors="coerce").fillna(0).astype(int).eq(1)
        | tls_error_text.str.contains(r"^ssl_", regex=True, na=False)
    ).astype(int)
    # Derive timeout flag from explicit flags and DNS or TLS timeout text
    enriched_df["timeout_flag"] = (
        pd.to_numeric(enriched_df.get("timeout_flag", 0), errors="coerce").fillna(0).astype(int).eq(1)
        | dns_error_text.str.contains("Timeout", case=False, regex=False, na=False)
        | tls_error_text.str.contains("timeout", case=False, regex=False, na=False)
    ).astype(int)
    # Set crawl status to unknown when enrichment was not attempted
    enriched_df["crawl_status"] = enriched_df.apply(
        lambda row: "unknown" if int(row.get("enrichment_attempted", 0)) == 0 else normalise_site_status_from_row(row),
        axis=1,
    )
    # Return the selectively enriched dataframe
    return enriched_df

## 7. Model Scoring

Live WHOISDS features are aligned with the saved training schema, then scored using the trained XGBoost model.

In [ ]:
# Define neutral defaults for live features unavailable in WHOISDS scoring
NEUTRAL_FEATURE_DEFAULTS = LIVE_NEUTRAL_DEFAULTS or {
    "cert_san_count": np.nan,
}

# Check whether a certificate subject common name matches the domain
def subject_cn_matches_domain(domain: str, subject_cn: Any) -> float:
    # Return missing when the subject common name is unavailable
    if not isinstance(subject_cn, str) or not subject_cn.strip():
        return np.nan
    # Normalise the subject common name and remove wildcard prefix
    cleaned_subject = normalise_domain(subject_cn.replace("*.", ""))
    # Return missing when the cleaned subject is unavailable
    if not cleaned_subject:
        return np.nan
    # Return one for exact or subdomain match and zero otherwise
    return float(domain == cleaned_subject or domain.endswith(f".{cleaned_subject}"))

# Build the live model feature matrix aligned to the saved training schema
def build_model_feature_matrix(df: pd.DataFrame) -> tuple[pd.DataFrame, Any, list[str]]:
    # Rebuild nameserver and status features from available live columns
    working = add_nameserver_features(df.copy(), NS_COLUMNS)
    working = add_status_features(working, STATUS_COLUMNS, WHOISDS_FIELD_MAP.get("status"))
    # Initialise the model feature dataframe and neutral-fill tracker
    X_live_model = pd.DataFrame(index=working.index)
    neutral_fills: list[str] = []
    # Define operational features required before model scoring
    required_operational_features = [
        "domain_length",
        "digit_count",
        "hyphen_count",
        "subdomain_count",
        "shannon_entropy",
        "nameserver_count",
        "status_count",
    ]
    # Detect missing operational features
    missing_operational_features = [
        feature for feature in required_operational_features if feature not in working.columns
    ]
    # Stop if required operational features are missing
    if missing_operational_features:
        raise ValueError(
            "Live model feature adapter is missing required operational features: "
            f"{sorted(missing_operational_features)}"
        )
    # Copy operational features into the model matrix
    for feature in required_operational_features:
        X_live_model[feature] = working[feature]
    # Map live enrichment columns to Notebook 2 model feature names
    live_to_model_mappings = [
        ("dns_has_a", "has_a_record"),
        ("dns_a_count", "a_record_count"),
        ("dns_aaaa_count", "aaaa_record_count"),
        ("dns_mx_count", "mx_record_count"),
        ("dns_ns_count", "ns_record_count"),
        ("dns_txt_count", "txt_record_count"),
        ("dns_has_soa", "has_soa_record"),
        ("dns_has_soa", "soa_present"),
        ("tls_days_until_expiry", "cert_days_until_expiry"),
        ("tls_validity_days", "cert_validity_days"),
    ]
    # Detect missing live enrichment source columns
    missing_live_sources = [
        live_column for live_column, _ in live_to_model_mappings if live_column not in working.columns
    ]
    # Stop if required live enrichment columns are missing
    if missing_live_sources:
        raise ValueError(
            "Live model feature adapter is missing required live enrichment columns: "
            f"{sorted(set(missing_live_sources))}"
        )
    # Copy mapped live enrichment features into model feature names
    for live_column, feature in live_to_model_mappings:
        X_live_model[feature] = working[live_column]
    # Check required live columns used for derived TLS and DNS features
    for required_live_column in [
        "dns_lookup_success",
        "tls_handshake_success",
        "tls_subject_common_name",
        "tls_issuer_common_name",
        "tls_not_before",
        "tls_not_after",
        "domain",
    ]:
        if required_live_column not in working.columns:
            raise ValueError(
                "Live model feature adapter is missing required live column "
                f"'{required_live_column}'."
            )
    # Derive DNS lookup failure from DNS lookup success
    dns_lookup_success = pd.to_numeric(working["dns_lookup_success"], errors="coerce")
    X_live_model["dns_lookup_failed"] = dns_lookup_success.fillna(0).eq(0).astype(int)
    # Carry TLS handshake success into the model matrix
    tls_handshake_success = pd.to_numeric(working["tls_handshake_success"], errors="coerce")
    X_live_model["tls_handshake_success"] = tls_handshake_success
    # Prepare certificate fields for certificate presence detection
    subject_cn_series = working["tls_subject_common_name"].fillna("").astype(str)
    issuer_cn_series = working["tls_issuer_common_name"].fillna("").astype(str)
    not_before_series = pd.to_datetime(working["tls_not_before"], errors="coerce")
    not_after_series = pd.to_datetime(working["tls_not_after"], errors="coerce")
    # Infer whether a certificate is present from subject, issuer, or validity dates
    cert_present_signal = (
        subject_cn_series.str.strip().ne("")
        | issuer_cn_series.str.strip().ne("")
        | not_before_series.notna()
        | not_after_series.notna()
    )
    X_live_model["cert_present"] = cert_present_signal.astype(int)
    # Derive certificate expiry flag from days until expiry
    cert_days_until_expiry = pd.to_numeric(
        X_live_model["cert_days_until_expiry"],
        errors="coerce",
    )
    X_live_model["cert_expired"] = np.where(
        cert_days_until_expiry.notna(),
        cert_days_until_expiry.lt(0).astype(float),
        np.nan,
    )
    # Derive certificate subject-domain match feature
    X_live_model["cert_subject_cn_matches_domain"] = [
        subject_cn_matches_domain(domain, subject)
        for domain, subject in zip(
            working["domain"].fillna("").astype(str),
            subject_cn_series,
        )
    ]
    # Use live SAN count when available or a neutral default otherwise
    if "cert_san_count" in working.columns:
        X_live_model["cert_san_count"] = working["cert_san_count"]
    else:
        X_live_model["cert_san_count"] = NEUTRAL_FEATURE_DEFAULTS["cert_san_count"]
        neutral_fills.append("cert_san_count")
    # Derive TLS scan failure from TLS handshake status
    X_live_model["tls_scan_failed"] = np.where(
        pd.notna(tls_handshake_success),
        tls_handshake_success.fillna(0).eq(0).astype(int),
        np.nan,
    )
    # Derive missing certificate flag when TLS succeeds but no certificate signal is present
    X_live_model["tls_no_certificate"] = np.where(
        pd.to_numeric(X_live_model["tls_handshake_success"], errors="coerce").fillna(0).eq(1),
        pd.to_numeric(X_live_model["cert_present"], errors="coerce").fillna(0).eq(0).astype(float),
        np.nan,
    )
    # Detect missing features against the saved model schema
    missing_model_features = [
        feature for feature in SAVED_MODEL_FEATURES if feature not in X_live_model.columns
    ]
    # Stop if live features do not match the saved schema
    if missing_model_features:
        raise ValueError(
            "Live model feature schema mismatch. "
            f"Missing: {sorted(missing_model_features)}."
        )
    # Reorder live features to match the saved model schema
    X_live_model = X_live_model[SAVED_MODEL_FEATURES].copy()
    assert list(X_live_model.columns) == SAVED_MODEL_FEATURES, (
        "X_live_model column order drifted from model_feature_schema.json"
    )
    # Coerce all model features to numeric values
    for feature in SAVED_MODEL_FEATURES:
        X_live_model[feature] = pd.to_numeric(X_live_model[feature], errors="coerce")
    # Check that no text columns remain in the model feature matrix
    invalid_object_columns = X_live_model.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()
    if invalid_object_columns:
        raise ValueError(
            "Live model feature matrix still contains non-numeric columns: "
            f"{invalid_object_columns}"
        )
    # Apply the saved preprocessor when available
    if preprocessor is not None:
        X_base = preprocessor.transform(X_live_model)
    else:
        X_base = X_live_model.to_numpy(dtype=float)
    # Return raw aligned features, transformed base matrix, and neutral-fill fields
    return X_live_model, X_base, sorted(dict.fromkeys(neutral_fills))

# Score live domains with the selected XGBoost model
def score_domains_with_model(df: pd.DataFrame) -> pd.DataFrame:
    # Work on a copy before adding scoring columns
    scored = df.copy()
    # Build model-ready features and use the base XGBoost input matrix
    _, X_base, neutral_fills = build_model_feature_matrix(scored)
    X_model = X_base
    # Validate final model input width against saved feature names
    expected_feature_count = len(MODEL_INPUT_FEATURE_NAMES)
    if hasattr(X_model, "shape") and X_model.shape[1] != expected_feature_count:
        raise ValueError(
            "Model input schema mismatch: expected "
            f"{expected_feature_count} features but built {X_model.shape[1]}"
        )
    # Generate phishing-risk probabilities
    model_score = xgb_model.predict_proba(X_model)[:, 1]
    # Attach scoring outputs and threshold metadata
    scored["model_score"] = model_score
    scored["threshold_used"] = THRESHOLD_USED
    scored["threshold_source"] = THRESHOLD_SOURCE
    scored["fallback_threshold_used"] = int(FALLBACK_THRESHOLD_USED)
    scored["final_model_name"] = FINAL_MODEL_NAME
    scored["model_schema_neutral_fills"] = "; ".join(neutral_fills) if neutral_fills else ""
    # Return scored live domains
    return scored

# Convert raw SHAP output into a consistent SHAP Explanation object
def to_shap_explanation(raw_explanation: Any, data: Any, feature_names: list[str]) -> shap.Explanation:
    # Extract SHAP values and base values
    values = raw_explanation.values
    base_values = raw_explanation.base_values
    # Select positive-class SHAP values for binary class-wise outputs
    if values.ndim == 3:
        values = values[:, :, 1]
        if np.asarray(base_values).ndim == 2:
            base_values = np.asarray(base_values)[:, 1]
    # Return the standardised SHAP explanation object
    return shap.Explanation(
        values=values,
        base_values=base_values,
        data=data,
        feature_names=feature_names,
    )

# Format the strongest positive or negative SHAP contributors
def top_shap_features(values: np.ndarray, feature_names: list[str], positive: bool, n: int = MAX_LOCAL_SHAP_FEATURES) -> str:
    order = np.argsort(values)
    if positive:
        order = order[::-1]
    # Collect contributors matching the requested direction
    rows = []
    for feature_idx in order:
        value = float(values[feature_idx])
        # Skip values with the opposite contribution direction
        if positive and value <= 0:
            continue
        if not positive and value >= 0:
            continue
        # Add formatted feature contribution
        rows.append(f"{feature_names[feature_idx]} ({value:.3f})")
        # Stop after the requested number of features
        if len(rows) == n:
            break
    # Return formatted contributors or none
    return "; ".join(rows) if rows else "none"

# Build SHAP plots and explanation tables for live scoring outputs
def build_live_scoring_shap_outputs(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Define output columns for top-domain SHAP explanations
    top_domain_columns = [
        "domain",
        "triage_tier",
        "triage_rank",
        "combined_triage_score",
        "model_score",
        "model_shap_top_positive_features",
        "model_shap_top_negative_features",
    ]
    # Define output columns for global feature importance summary
    feature_summary_columns = ["feature", "mean_abs_shap"]
    # Return empty SHAP outputs when there are no rows
    if df.empty:
        return pd.DataFrame(columns=feature_summary_columns), pd.DataFrame(columns=top_domain_columns)
    # Build model inputs and initialise SHAP explainer
    try:
        _, X_base, _ = build_model_feature_matrix(df)
        X_model = X_base
        explainer = shap.TreeExplainer(xgb_model)
    except Exception as exc:
        warnings.warn(f"SHAP output skipped: {exc}")
        return pd.DataFrame(columns=feature_summary_columns), pd.DataFrame(columns=top_domain_columns)
    # Use the saved final model input feature names
    shap_feature_names = MODEL_INPUT_FEATURE_NAMES

    # Sample rows for global SHAP plots when the live set is large
    rng = np.random.default_rng(42)
    if X_model.shape[0] > MAX_SHAP_GLOBAL_ROWS:
        shap_global_idx = np.sort(rng.choice(X_model.shape[0], size=MAX_SHAP_GLOBAL_ROWS, replace=False))
    else:
        shap_global_idx = np.arange(X_model.shape[0])
    # Calculate global SHAP explanations on sampled rows
    X_shap_global = X_model[shap_global_idx]
    shap_global = to_shap_explanation(explainer(X_shap_global), X_shap_global, shap_feature_names)
    # Plot global SHAP beeswarm summary
    shap.plots.beeswarm(shap_global, max_display=MAX_SHAP_DISPLAY_FEATURES, show=False)
    plt.title(f"SHAP Summary: {FINAL_MODEL_NAME}")
    plt.tight_layout()
    plt.show()
    # Plot global SHAP feature importance bars
    shap.plots.bar(shap_global, max_display=MAX_SHAP_DISPLAY_FEATURES, show=False)
    plt.title(f"Top SHAP Features: {FINAL_MODEL_NAME}")
    plt.tight_layout()
    plt.show()
    # Build mean absolute SHAP feature importance table
    mean_abs_shap = np.abs(shap_global.values).mean(axis=0)
    feature_summary_df = (
        pd.DataFrame({"feature": shap_feature_names, "mean_abs_shap": mean_abs_shap})
        .sort_values(["mean_abs_shap", "feature"], ascending=[False, True])
        .head(MAX_SHAP_DISPLAY_FEATURES)
        .reset_index(drop=True)
    )
    # Select top live domains for local SHAP explanations
    top_rows = (
        df.sort_values(["combined_triage_score", "model_score", "domain"], ascending=[False, False, True])
        .head(MAX_TOP_SHAP_REVIEW_ROWS)
        .copy()
    )
    # Map selected top rows back to their model matrix positions
    top_positions = df.index.get_indexer(top_rows.index)
    # Calculate local SHAP explanations for selected top rows
    shap_local = to_shap_explanation(explainer(X_model[top_positions]), X_model[top_positions], shap_feature_names)
    # Build top-domain explanation rows
    local_rows = []
    for local_pos, (_, row) in enumerate(top_rows.iterrows()):
        values = shap_local.values[local_pos]
        # Store model score, triage fields, and top SHAP contributors
        local_rows.append({
            "domain": row.get("domain", ""),
            "triage_tier": row.get("triage_tier", ""),
            "triage_rank": row.get("triage_rank", np.nan),
            "combined_triage_score": row.get("combined_triage_score", np.nan),
            "model_score": row.get("model_score", np.nan),
            "model_shap_top_positive_features": top_shap_features(values, shap_feature_names, positive=True),
            "model_shap_top_negative_features": top_shap_features(values, shap_feature_names, positive=False),
        })
    # Return global feature summary and local top-domain explanation table
    top_domain_shap_df = pd.DataFrame(local_rows, columns=top_domain_columns)
    return feature_summary_df, top_domain_shap_df

## 8. Triage Scoring

This stage combines the model probability with live contextual, infrastructure, crawler, and uncertainty evidence to produce an analyst-facing priority score for each candidate domain. It assigns weighted score components for keyword signals, DNS and TLS availability, reachable webpages, login or form indicators, crawler evidence strength, and missing or weak evidence. The resulting score is then used to assign a triage tier for prioritised analyst review.

In [ ]:
# Define triage tier order and score calibration settings
TIER_SORT_ORDER = ["High", "Medium", "Low"]
MODEL_SCORE_CLIP_MIN = 0.05
MODEL_SCORE_CLIP_MAX = 0.95
COMBINATION_INTERCEPT = -1.35
COMBINATION_W_ML = 0.55
COMBINATION_W_CONTEXT = 1.55
COMBINATION_W_CRAWLER = 0.055
COMBINATION_W_UNCERTAINTY = 1.45

# Define crawler evidence point values and tier thresholds
CRAWLER_EVIDENCE_LIMITED_POINTS = 5.0
CRAWLER_EVIDENCE_MODERATE_POINTS = 12.5
CRAWLER_EVIDENCE_STRONG_POINTS = 22.5
HIGH_TRIAGE_THRESHOLD = 80.0
MEDIUM_TRIAGE_THRESHOLD = 55.0

# Define contextual priority groups used for triage
HIGH_PRIORITY_CATEGORY_COLUMNS = {
    "Malta GOV": "malta_gov_flag",
    "Typosquat": "typosquatting_flag",
}
MEDIUM_PRIORITY_CATEGORY_COLUMNS = {
    "Malta": "malta_flag",
    "Bank": "bank_flag",
}
LOW_PRIORITY_CATEGORY_COLUMNS = {
    "Generic GOV": "generic_gov_flag",
    "Finance": "finance_keyword_flag",
    "Account": "account_keyword_flag",
}

# Define suspicious and placeholder page text indicators
SUSPICIOUS_PAGE_TERMS = [
    "login",
    "sign in",
    "signin",
    "verify",
    "account",
    "secure",
    "payment",
    "bank",
    "card",
    "wallet",
    "auth",
    "password",
    "invoice",
    "refund",
]
PLACEHOLDER_PAGE_TERMS = [
    "coming soon",
    "under construction",
    "parked",
    "buy this domain",
    "domain for sale",
    "website builder",
    "default page",
    "placeholder",
]

# Return a numeric series for a dataframe column or zeros when missing
def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    # Coerce an existing column to numeric values
    if column in df.columns:
        return pd.to_numeric(df[column], errors="coerce").fillna(0)
    # Return zeros when the column is unavailable
    return pd.Series(0, index=df.index, dtype="float64")

# Convert a scalar value to float with a default fallback
def scalar_float(value: Any, default: float = 0.0) -> float:
    # Coerce the value to numeric form
    coerced = pd.to_numeric(value, errors="coerce")
    # Return the default when coercion fails
    if pd.isna(coerced):
        return float(default)
    return float(coerced)

# Convert a scalar value to an integer with a default fallback
def scalar_int(value: Any, default: int = 0) -> int:
    return int(round(scalar_float(value, default)))

# Clean a scalar value into stripped text
def clean_text_value(value: Any) -> str:
    # Return empty text for missing values
    if value is None:
        return ""
    if pd.isna(value):
        return ""
    return str(value).strip()

# Compact text for broad keyword matching
def compact_keyword_text(value: Any) -> str:
    # Remove non-alphanumeric characters from lowercase text
    return re.sub(r"[^a-z0-9]+", "", clean_text_value(value).lower())

# Tokenise text into lowercase alphanumeric tokens
def tokenise_text_value(value: Any) -> set[str]:
    # Return unique non-empty tokens
    return {
        token
        for token in re.split(r"[^a-z0-9]+", clean_text_value(value).lower())
        if token
    }

# Normalise model probabilities into the zero-to-one range
def normalise_model_probability(series: pd.Series) -> pd.Series:
    # Coerce values to numeric probabilities and remove negative values
    values = pd.to_numeric(series, errors="coerce").fillna(0.0).clip(lower=0.0)
    # Convert percentage  scores to probabilities
    if values.max() > 1.0:
        values = values / 100.0
    # Cap probabilities at one
    return values.clip(upper=1.0)

# Apply the sigmoid transform
def sigmoid(value: Any) -> Any:
    # Convert log odds to probability values
    return 1.0 / (1.0 + np.exp(-value))

# Convert probabilities to clipped log odds
def safe_logit(series: pd.Series) -> pd.Series:
    # Clip probabilities away from zero and one for stable logit values
    clipped = pd.to_numeric(series, errors="coerce").fillna(0.5).clip(
        lower=MODEL_SCORE_CLIP_MIN,
        upper=MODEL_SCORE_CLIP_MAX,
    )
    # Return log odds
    return np.log(clipped / (1.0 - clipped))

# Summarise matched contextual priority flags for one row
def contextual_priority_state(row: pd.Series) -> dict[str, Any]:
    # Collect matched high, medium, and low priority labels
    matched_high_flags = [label for label, column in HIGH_PRIORITY_CATEGORY_COLUMNS.items() if scalar_int(row.get(column, 0)) == 1]
    matched_medium_flags = [label for label, column in MEDIUM_PRIORITY_CATEGORY_COLUMNS.items() if scalar_int(row.get(column, 0)) == 1]
    matched_low_flags = [label for label, column in LOW_PRIORITY_CATEGORY_COLUMNS.items() if scalar_int(row.get(column, 0)) == 1]
    return {
    # Return matched flag summaries and derived boolean indicators
        "matched_high_flags": "; ".join(matched_high_flags),
        "matched_medium_flags": "; ".join(matched_medium_flags),
        "matched_low_flags": "; ".join(matched_low_flags),
        "matched_high_flags_count": len(matched_high_flags),
        "matched_medium_flags_count": len(matched_medium_flags),
        "matched_low_flags_count": len(matched_low_flags),
        "has_high_priority_flag": int(bool(matched_high_flags)),
        "has_medium_priority_flag": int(bool(matched_medium_flags)),
        "has_low_priority_flag": int(bool(matched_low_flags)),
        "bank_with_high_combo_flag": int(bool(matched_high_flags and matched_medium_flags)),
        "low_with_medium_or_high_combo_flag": int(bool(matched_low_flags and (matched_medium_flags or matched_high_flags))),
    }

# Collect domain-like text fields from a row
def row_text_values(row: pd.Series) -> list[str]:
    # Collect non-empty values from domain, URL, and host-style columns
    values: list[str] = []
    for column in ["domain", "registered_domain", "fqdn", "hostname", "url", "final_url"]:
        value = clean_text_value(row.get(column, ""))
        if value:
            values.append(value)
    # Return text values used for keyword matching
    return values

# Check whether a keyword appears as a strict token in row text
def strict_keyword_match_in_row(row: pd.Series, keyword: str) -> bool:
    # Prepare the keyword for token matching
    target = clean_text_value(keyword).lower()
    if not target:
        return False
    # Search each row text value for an exact token match
    for value in row_text_values(row):
        if target in tokenise_text_value(value):
            return True
    # Return false when no strict match is found
    return False

# Check whether a keyword appears as a broad compact substring
def broad_substring_keyword_match_in_row(row: pd.Series, keyword: str) -> bool:
    # Prepare compact keyword text
    target = compact_keyword_text(keyword)
    if not target:
        return False
    # Avoid counting strict token matches as broad substring matches
    if strict_keyword_match_in_row(row, keyword):
        return False
    # Search compact row text values for the compact keyword
    for value in row_text_values(row):
        if target in compact_keyword_text(value):
            return True
    # Return false when no broad substring match is found
    return False

# Combine page title, meta description, and visible snippet into one text blob
def page_text_blob(row: pd.Series) -> str:
    parts = [
        clean_text_value(row.get("page_title", "")),
        clean_text_value(row.get("page_meta_description", "")),
        clean_text_value(row.get("visible_text_snippet", "")),
    ]
    # Return lowercased combined page text
    return " ".join(part for part in parts if part).strip().lower()

# Detect suspicious terms in page text
def page_keyword_signal_flag_from_row(row: pd.Series) -> int:
    # Build combined page text
    page_text = page_text_blob(row)
    # Return zero when no page text is available
    if not page_text:
        return 0
    # Return one when suspicious terms appear in page text
    return int(any(term in page_text for term in SUSPICIOUS_PAGE_TERMS))

# Detect placeholder or parked page content
def placeholder_page_flag_from_row(row: pd.Series) -> int:
    # Read site or crawl status text
    status_text = clean_text_value(row.get("site_status", row.get("crawl_status", ""))).lower()
    # Build combined page text
    page_text = page_text_blob(row)
    # Use status first when it directly indicates placeholder content
    if "parked" in status_text or "placeholder" in status_text:
        return 1
    # Return one when placeholder terms appear in page text
    return int(any(term in page_text for term in PLACEHOLDER_PAGE_TERMS))

# Check whether crawler evidence contains usable page content
def has_page_content(row: pd.Series) -> bool:
    # Trust the explicit content flag when present
    if scalar_int(row.get("content_detected", 0)) == 1:
        return True
    # Fall back to non-empty page text
    return bool(page_text_blob(row))

# Evaluate contextual domain evidence for triage scoring
def evaluate_contextual_evidence(row: pd.Series) -> dict[str, Any]:
    # Read contextual keyword flags
    malta_gov_flag = scalar_int(row.get("malta_gov_flag", 0)) == 1
    malta_flag = scalar_int(row.get("malta_flag", 0)) == 1
    bank_flag = scalar_int(row.get("bank_flag", 0)) == 1
    typosquat_flag = scalar_int(row.get("typosquatting_flag", 0)) == 1
    generic_gov_flag = scalar_int(row.get("generic_gov_flag", 0)) == 1
    finance_flag = scalar_int(row.get("finance_keyword_flag", 0)) == 1
    account_flag = scalar_int(row.get("account_keyword_flag", 0)) == 1
    # Detect strict high-confidence Malta government terms
    strict_mita_flag = strict_keyword_match_in_row(row, "mita")
    strict_lesa_flag = strict_keyword_match_in_row(row, "lesa")
    strict_identita_flag = strict_keyword_match_in_row(row, "identita")
    strict_gov_brand_flag = any(
        strict_keyword_match_in_row(row, keyword)
        or keyword in " ".join(row_text_values(row)).lower()
        for keyword in ["gov.mt", "govmt", "mtgov", "gov-mt", "mt-gov", "govmalta"]
    )
    # Detect broader Malta government substrings
    broad_mita_substring_flag = broad_substring_keyword_match_in_row(row, "mita")
    broad_lesa_substring_flag = broad_substring_keyword_match_in_row(row, "lesa")
    broad_identita_substring_flag = broad_substring_keyword_match_in_row(row, "identita")
    broad_substring_flag = broad_mita_substring_flag or broad_lesa_substring_flag or broad_identita_substring_flag
    # Build combined context flags
    weak_generic_flag = generic_gov_flag or finance_flag or account_flag
    high_support_flag = malta_gov_flag or malta_flag or strict_mita_flag or strict_lesa_flag or strict_identita_flag or strict_gov_brand_flag
    typosquat_supported_flag = typosquat_flag and (high_support_flag or bank_flag)
    malta_relevance_flag = int(
        malta_gov_flag
        or typosquat_supported_flag
        or (malta_flag and (bank_flag or generic_gov_flag or finance_flag or account_flag))
    )
    weak_generic_only_flag = int(weak_generic_flag and not (high_support_flag or bank_flag or typosquat_flag))
    # Initialise context evidence score
    context_score = 0.0
    # Score strongest contextual evidence first
    if strict_mita_flag or strict_lesa_flag or strict_identita_flag or strict_gov_brand_flag:
        context_score = 1.95
    elif malta_gov_flag and not broad_substring_flag:
        context_score = 1.80
    elif malta_flag and bank_flag:
        context_score = 1.55
    elif typosquat_supported_flag:
        context_score = 1.60
    elif malta_flag:
        context_score = 1.15
    elif bank_flag:
        context_score = 1.00
    elif typosquat_flag:
        context_score = 0.90
    elif weak_generic_flag:
        context_score = 0.35
    # Raise context score for supported Malta or combined sector evidence
    if malta_relevance_flag == 1:
        context_score = max(context_score, 1.45)
    if bank_flag and high_support_flag:
        context_score = max(context_score, 2.00)
    if weak_generic_flag and (bank_flag or high_support_flag):
        context_score = max(context_score, 1.85)
    # Penalise broad substring-only matches that lack strict support
    if broad_substring_flag:
        if not (strict_mita_flag or strict_lesa_flag or strict_identita_flag or strict_gov_brand_flag):
            context_score = min(context_score, 0.70 if (bank_flag or malta_flag or typosquat_flag) else 0.45)
    # Return contextual evidence flags and score
    return {
        "strict_mita_flag": int(strict_mita_flag),
        "strict_lesa_flag": int(strict_lesa_flag),
        "strict_identita_flag": int(strict_identita_flag),
        "strict_gov_brand_flag": int(strict_gov_brand_flag),
        "broad_mita_substring_flag": int(broad_mita_substring_flag),
        "broad_lesa_substring_flag": int(broad_lesa_substring_flag),
        "broad_identita_substring_flag": int(broad_identita_substring_flag),
        "broad_substring_flag": int(broad_substring_flag),
        "malta_relevance_flag": malta_relevance_flag,
        "typosquat_supported_flag": int(typosquat_supported_flag),
        "weak_generic_only_flag": weak_generic_only_flag,
        "context_evidence_score": float(context_score),
    }
    
# Evaluate crawler evidence for triage scoring
def evaluate_crawler_evidence(row: pd.Series) -> dict[str, Any]:
    http_status = scalar_int(row.get("http_status", 0))
    redirect_detected = scalar_int(row.get("redirect_detected_flag", 0)) == 1
    credential_form_detected = scalar_int(row.get("credential_form_detected_flag", 0)) == 1
    html_form_count = scalar_int(row.get("html_form_count", 0))
    html_input_count = scalar_int(row.get("html_input_count", 0))
    page_keyword_signal_flag = page_keyword_signal_flag_from_row(row)
    placeholder_page_flag = placeholder_page_flag_from_row(row)
    active_http_flag = http_status in {200, 301, 302}
    page_content_flag = has_page_content(row)
    dns_error_flag = scalar_int(row.get("dns_error_flag", 0)) == 1
    timeout_flag = scalar_int(row.get("timeout_flag", 0)) == 1
    ssl_error_flag = scalar_int(row.get("ssl_error_flag", 0)) == 1
    has_form_elements = html_form_count > 0 or html_input_count > 0
    # Check whether contextual signals support crawler evidence
    contextual_support_flag = any([
        scalar_int(row.get("malta_gov_flag", 0)) == 1,
        scalar_int(row.get("malta_flag", 0)) == 1,
        scalar_int(row.get("bank_flag", 0)) == 1,
        scalar_int(row.get("typosquatting_flag", 0)) == 1,
        scalar_int(row.get("generic_gov_flag", 0)) == 1,
        scalar_int(row.get("finance_keyword_flag", 0)) == 1,
        scalar_int(row.get("account_keyword_flag", 0)) == 1,
    ])
    # Initialise crawler evidence outputs
    crawler_evidence_score = 0.0
    crawler_evidence_level = "absent_crawler_evidence"
    # Score crawler evidence when error and placeholder signals are absent
    if not (dns_error_flag or timeout_flag or ssl_error_flag or placeholder_page_flag):
        if (
            credential_form_detected
            or (has_form_elements and page_keyword_signal_flag == 1)
            or (redirect_detected and page_keyword_signal_flag == 1 and contextual_support_flag)
        ):
            crawler_evidence_level = "strong_crawler_evidence"
            crawler_evidence_score = CRAWLER_EVIDENCE_STRONG_POINTS
        elif (
            page_keyword_signal_flag == 1
            or redirect_detected
            or has_form_elements
            or (active_http_flag and page_content_flag and contextual_support_flag)
        ):
            crawler_evidence_level = "moderate_crawler_evidence"
            crawler_evidence_score = CRAWLER_EVIDENCE_MODERATE_POINTS
        elif active_http_flag or page_content_flag:
            crawler_evidence_level = "limited_crawler_evidence"
            crawler_evidence_score = CRAWLER_EVIDENCE_LIMITED_POINTS
    # Derive crawler evidence flags
    crawler_evidence_score = float(crawler_evidence_score)
    crawler_evidence_flag = int(crawler_evidence_score > 0)
    phishing_content_detected_flag = int(page_keyword_signal_flag == 1 or credential_form_detected or redirect_detected)
    # Return crawler evidence outputs
    return {
        "active_http_flag": int(active_http_flag),
        "page_content_flag": int(page_content_flag),
        "page_keyword_signal_flag": int(page_keyword_signal_flag),
        "placeholder_page_flag": int(placeholder_page_flag),
        "crawler_evidence_level": crawler_evidence_level,
        "crawler_evidence_score": crawler_evidence_score,
        "crawler_evidence_flag": crawler_evidence_flag,
        "phishing_content_detected_flag": phishing_content_detected_flag,
    }

# Calculate uncertainty and reliability penalties for triage scoring
def evaluate_penalty_score(row: pd.Series) -> float:
    # Initialise the penalty score
    penalty = 0.0
    # Add penalties for weaker contextual or unreliable crawler signals
    if scalar_int(row.get("broad_substring_flag", 0)) == 1:
        penalty += 0.80
    if scalar_int(row.get("weak_generic_only_flag", 0)) == 1:
        penalty += 0.60
    if scalar_int(row.get("crawler_evidence_flag", 0)) == 0:
        penalty += 0.20
    if scalar_int(row.get("placeholder_page_flag", 0)) == 1:
        penalty += 0.75
    if scalar_int(row.get("dns_error_flag", 0)) == 1:
        penalty += 0.85
    if scalar_int(row.get("timeout_flag", 0)) == 1:
        penalty += 0.90
    if scalar_int(row.get("ssl_error_flag", 0)) == 1:
        penalty += 0.40
    if scalar_int(row.get("active_http_flag", 0)) == 1 and scalar_int(row.get("page_content_flag", 0)) == 0:
        penalty += 0.55
    # Cap and return the final penalty score
    return float(min(penalty, 2.50))

# Apply a row function with progress reporting
def apply_row_mapping_with_progress(df: pd.DataFrame, row_func: Any, label: str) -> pd.DataFrame:
    total = len(df)
    if total == 0:
        return pd.DataFrame(index=df.index)
    # Collect row-level function outputs
    records: list[dict[str, Any]] = []
    checkpoint = max(1, total // 20)
    # Apply the row function and print progress checkpoints
    for index, (_, row) in enumerate(df.iterrows(), start=1):
        records.append(dict(row_func(row)))
        if index == total or index % checkpoint == 0:
            progress_pct = (index / total) * 100.0
            print(f"{label}: {index:,}/{total:,} ({progress_pct:5.1f}%)")
    # Return row outputs aligned to the original dataframe index
    return pd.DataFrame.from_records(records, index=df.index)

# Identify rows with high or medium matched contextual flags
def matched_high_medium_mask(df: pd.DataFrame) -> pd.Series:
    # Detect populated high and medium matched flag fields
    matched_high = df.get("matched_high_flags", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().ne("")
    matched_medium = df.get("matched_medium_flags", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().ne("")
    # Return rows matching either high or medium context
    return matched_high | matched_medium

# Prepare contextual matched flags before selective enrichment
def prepare_matched_flag_queue(df: pd.DataFrame) -> pd.DataFrame:
    # Work on a copy before adding queue fields
    queued = df.copy()
    # Build contextual priority state for each row
    contextual_state_df = apply_row_mapping_with_progress(
        queued,
        contextual_priority_state,
        "Matched flags (priority groups)",
    )
    # Attach contextual priority columns to the queue
    for column in contextual_state_df.columns:
        queued[column] = contextual_state_df[column]
    # Build readable matched flag text
    queued["matched_flags"] = queued.apply(build_matched_flags, axis=1)
    # Mark rows eligible for selective enrichment
    queued["crawl_candidate_flag"] = matched_high_medium_mask(queued).astype(int)
    # Return the queued dataframe
    return queued

# Apply final operational triage scoring
def apply_operational_scoring(
    df: pd.DataFrame,
    probability_column: str = "model_score",
) -> tuple[pd.DataFrame, dict[str, int]]:
    # Work on a copy before adding triage outputs
    scored = df.copy()
    # Read model probability scores or use zeros when unavailable
    probability_input = (
        scored[probability_column]
        if probability_column in scored.columns
        else pd.Series(0.0, index=scored.index, dtype="float64")
    )
    # Normalise probabilities and convert them to ML evidence log odds
    scored["model_score"] = normalise_model_probability(probability_input).round(6)
    scored["ml_evidence_score"] = safe_logit(scored["model_score"]).round(6)
    # Derive crawler helper flags from redirect and password input counts
    scored["redirect_detected_flag"] = numeric_series(scored, "redirect_count").gt(0).astype(int)
    scored["credential_form_detected_flag"] = numeric_series(scored, "html_password_input_count").gt(0).astype(int)
    # Add contextual priority group summaries
    contextual_state_df = apply_row_mapping_with_progress(
        scored,
        contextual_priority_state,
        "Triage scoring (priority groups)",
    )
    for column in contextual_state_df.columns:
        scored[column] = contextual_state_df[column]
    # Add contextual evidence scores
    contextual_evidence_df = apply_row_mapping_with_progress(
        scored,
        evaluate_contextual_evidence,
        "Triage scoring (context evidence)",
    )
    for column in contextual_evidence_df.columns:
        scored[column] = contextual_evidence_df[column]
    # Add crawler evidence scores
    crawler_evidence_df = apply_row_mapping_with_progress(
        scored,
        evaluate_crawler_evidence,
        "Triage scoring (crawler evidence)",
    )
    for column in crawler_evidence_df.columns:
        scored[column] = crawler_evidence_df[column]
    # Calculate row-level penalty scores with periodic progress output
    penalty_values: list[float] = []
    penalty_total = len(scored)
    for index, row in enumerate(scored.itertuples(index=False), start=1):
        penalty_values.append(float(evaluate_penalty_score(pd.Series(row._asdict()))))
        if penalty_total > 0 and (index == penalty_total or index == 1 or index % 10000 == 0):
            progress_pct = 100.0 * index / penalty_total
            print(f"Triage scoring (uncertainty): {index:,}/{penalty_total:,} ({progress_pct:5.1f}%)")
    # Store the final penalty score
    scored["penalty_score"] = pd.Series(penalty_values, index=scored.index, dtype="float64").round(6)
    # Combine ML, context, crawler, and uncertainty signals into one triage score
    combination_log_odds = (
        COMBINATION_INTERCEPT
        + (COMBINATION_W_ML * scored["ml_evidence_score"])
        + (COMBINATION_W_CONTEXT * pd.to_numeric(scored["context_evidence_score"], errors="coerce").fillna(0.0))
        + (COMBINATION_W_CRAWLER * pd.to_numeric(scored["crawler_evidence_score"], errors="coerce").fillna(0.0))
        - (COMBINATION_W_UNCERTAINTY * pd.to_numeric(scored["penalty_score"], errors="coerce").fillna(0.0))
    )
    scored["combined_triage_score"] = (100.0 * sigmoid(combination_log_odds)).clip(lower=0.0, upper=100.0).round(2)
    # Assign High, Medium, or Low triage tier from score thresholds
    scored["triage_tier"] = np.select(
        [
            scored["combined_triage_score"].ge(HIGH_TRIAGE_THRESHOLD),
            scored["combined_triage_score"].ge(MEDIUM_TRIAGE_THRESHOLD),
        ],
        [
            "High",
            "Medium",
        ],
        default="Low",
    )
    # Recalculate crawl candidate flag after triage feature updates
    scored["crawl_candidate_flag"] = matched_high_medium_mask(scored).astype(int)
    # Sort by triage priority and assign a review rank
    scored["triage_tier"] = pd.Categorical(
        scored["triage_tier"],
        categories=TIER_SORT_ORDER,
        ordered=True,
    )
    scored = scored.sort_values(
        by=["combined_triage_score", "model_score", "domain"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    scored["triage_rank"] = pd.RangeIndex(start=1, stop=len(scored) + 1)
    scored["triage_tier"] = scored["triage_tier"].astype(str)
    # Build operational diagnostics for reporting
    diagnostics = {
        "high_tier_count": int(scored["triage_tier"].eq("High").sum()),
        "medium_tier_count": int(scored["triage_tier"].eq("Medium").sum()),
        "low_tier_count": int(scored["triage_tier"].eq("Low").sum()),
        "crawl_candidate_count": int(scored["crawl_candidate_flag"].sum()),
        "crawler_evidence_count": int(scored["crawler_evidence_flag"].sum()),
        "credential_form_count": int(scored["credential_form_detected_flag"].sum()),
        "redirect_detected_count": int(scored["redirect_detected_flag"].sum()),
    }
    # Return scored rows and diagnostic counts
    return scored, diagnostics

## 9. Export

In [ ]:
# Normalise HTTP status values into clean string codes
def normalise_http_status_value(value: Any) -> str:
    # Coerce the value to numeric form
    numeric = pd.to_numeric(value, errors="coerce")
    # Return empty text when no valid status is available
    if pd.isna(numeric):
        return ""
    # Return the rounded integer status as text
    try:
        return str(int(round(float(numeric))))
    except Exception:
        return ""

# Derive a standard site status from HTTP and connection signals
def normalise_site_status_from_row(row: pd.Series) -> str:
    # Read HTTP status and raw crawl status
    http_status = normalise_http_status_value(row.get("http_status", ""))
    raw_status = clean_text_value(row.get("site_status", row.get("crawl_status", ""))).lower()
    # Map known HTTP statuses to standard labels
    if http_status:
        if http_status == "525":
            return "ssl_handshake_failed"
        if 200 <= int(http_status) <= 299:
            return "active"
        if http_status in {"301", "302", "303", "307", "308"}:
            return "redirecting"
        if http_status == "401":
            return "auth_required"
        if http_status == "403":
            return "blocked_or_forbidden"
        if http_status == "404":
            return "not_found"
        if http_status == "410":
            return "gone"
        if http_status == "429":
            return "rate_limited"
        if http_status in {"500", "501", "502", "503", "504"}:
            return "server_error"
        return "unknown"
    # Fall back to DNS, timeout, SSL, and raw crawl status signals
    if scalar_int(row.get("dns_error_flag", 0)) == 1:
        return "dns_error"
    if scalar_int(row.get("timeout_flag", 0)) == 1:
        return "timeout"
    if scalar_int(row.get("ssl_error_flag", 0)) == 1:
        return "connection_error"
    if raw_status in {"connection_error", "ssl_error"}:
        return "connection_error"
    # Return unknown when no status signal is available
    return "unknown"

# Build a readable form status from crawler form counts
def build_form_status_from_row(row: pd.Series) -> str:
    # Read form, password, and input counts
    form_count = scalar_int(row.get("html_form_count", 0))
    password_count = scalar_int(row.get("html_password_input_count", 0))
    input_count = scalar_int(row.get("html_input_count", 0))
    # Build a detected form summary when form elements are present
    if form_count > 0 or password_count > 0 or input_count > 0:
        return _build_detected_form_summary(form_count, password_count, input_count)
    # Reuse an existing non-default form summary when available
    existing_summary = clean_text_value(row.get("detected_form_summary", ""))
    if existing_summary and existing_summary.lower() not in {"no forms detected", "form status unknown"}:
        return existing_summary
    # Check whether crawler context exists even without forms
    has_crawler_context = any([
        scalar_int(row.get("enrichment_attempted", 0)) == 1,
        bool(normalise_http_status_value(row.get("http_status", ""))),
        bool(clean_text_value(row.get("final_url", ""))),
        bool(clean_text_value(row.get("page_title", ""))),
        bool(clean_text_value(row.get("page_meta_description", ""))),
        bool(clean_text_value(row.get("visible_text_snippet", ""))),
        bool(clean_text_value(row.get("site_status", row.get("crawl_status", "")))),
    ])
    # Return a known no-form status or an unknown form status
    return "No forms detected" if has_crawler_context else "Form status unknown"

# Convert crawler evidence fields into a compact evidence strength label
def build_crawler_evidence_clause(row: pd.Series) -> str:
    # Read crawler evidence level and score
    crawler_level = clean_text_value(row.get("crawler_evidence_level", "")).lower()
    crawler_score = scalar_float(row.get("crawler_evidence_score", 0))
    # Return strong evidence for high score or credential form signals
    if crawler_level == "strong_crawler_evidence" or crawler_score >= CRAWLER_EVIDENCE_STRONG_POINTS or scalar_int(row.get("credential_form_detected_flag", 0)) == 1:
        return "strong"
    # Return moderate evidence for moderate score or level
    if crawler_level == "moderate_crawler_evidence" or crawler_score >= CRAWLER_EVIDENCE_MODERATE_POINTS:
        return "moderate"
    # Check for basic reachability evidence
    has_reachability_evidence = any([
        bool(normalise_http_status_value(row.get("http_status", ""))),
        bool(clean_text_value(row.get("final_url", ""))),
        bool(clean_text_value(row.get("page_title", ""))),
        bool(clean_text_value(row.get("visible_text_snippet", ""))),
        scalar_int(row.get("enrichment_attempted", 0)) == 1,
        clean_text_value(row.get("site_status", row.get("crawl_status", ""))).lower() not in {"", "unknown"},
    ])
    # Return limited evidence for low score, evidence flag, or reachability
    if crawler_level == "limited_crawler_evidence" or crawler_score >= CRAWLER_EVIDENCE_LIMITED_POINTS or scalar_int(row.get("crawler_evidence_flag", 0)) == 1 or has_reachability_evidence:
        return "limited"
    # Return absent when no crawler evidence is available
    return "absent"

# Build a compact analyst-facing triage explanation
def triage_explanation(row: pd.Series) -> str:
    # Initialise explanation parts
    details: list[str] = []
    # Resolve the primary matched keyword
    matched_keyword = clean_text_value(row.get("matched_keyword", ""))
    if not matched_keyword:
        matched_keywords = clean_text_value(row.get("matched_keywords", ""))
        if not matched_keywords:
            domain_value = clean_text_value(row.get("domain", ""))
            matched_keywords = extract_matched_keywords(domain_value) if domain_value else ""
        matched_keyword_values = [part.strip() for part in matched_keywords.split(";") if part.strip()]
        matched_keyword = matched_keyword_values[0] if matched_keyword_values else ""
    # Add matched keyword
    if matched_keyword:
        details.append(f"Matched Keyword: '{matched_keyword}'")
    # Resolve keyword risk flag from explicit field or matched flag columns
    keyword_risk_flag = clean_text_value(row.get("keyword_risk_flag", ""))
    if not keyword_risk_flag:
        for severity, column in [("HIGH", "matched_high_flags"), ("MEDIUM", "matched_medium_flags"), ("LOW", "matched_low_flags")]:
            raw_value = clean_text_value(row.get(column, "")).replace("; ", ", ")
            categories = [part.strip() for part in raw_value.split(",") if part.strip()]
            if categories:
                keyword_risk_flag = f"{severity}, {categories[0]}"
                break
    # Add keyword risk flag detail
    if keyword_risk_flag:
        severity, category = keyword_risk_flag.split(", ", 1)
        details.append(f"Keyword Risk Flag: '{severity}', '{category}'")
    # Add HTTP and site status detail
    http_status = normalise_http_status_value(row.get("http_status", ""))
    site_status = clean_text_value(row.get("site_status", "")) or normalise_site_status_from_row(row)
    details.append(f"HTTP {http_status}, {site_status}" if http_status else f"HTTP unavailable, {site_status}")
    # Add final URL availability detail
    final_url = clean_text_value(row.get("final_url", ""))
    details.append(clean_text_value(row.get("final_url_status", "")) or ("final URL available" if final_url else "final URL unavailable"))
    # Add no-content note when crawler context exists but no text was extracted
    has_title = bool(clean_text_value(row.get("page_title", "")))
    has_visible_text = bool(clean_text_value(row.get("visible_text_snippet", "")))
    if not (has_title or has_visible_text):
        has_crawler_context = any([
            scalar_int(row.get("enrichment_attempted", 0)) == 1,
            bool(http_status),
            bool(final_url),
            bool(clean_text_value(row.get("page_meta_description", ""))),
            site_status.lower() not in {"", "unknown"},
        ])
        if has_crawler_context:
            details.append("no title or visible text extracted")
    # Add form status and crawler evidence strength
    form_status = build_form_status_from_row(row)
    details.append(form_status[0].lower() + form_status[1:] if form_status else "form status unknown")
    details.append(f"{build_crawler_evidence_clause(row)} crawler evidence.")
    # Return unique explanation parts joined for analyst review
    return "; ".join(dict.fromkeys(part for part in details if part))

# Build a compact matched flag summary from priority columns
def build_matched_flags(row: pd.Series) -> str:
    # Read high, medium, and low matched flag values
    parts: list[str] = []
    high_flags = str(row.get("matched_high_flags", "") or "").replace("; ", ", ").strip()
    medium_flags = str(row.get("matched_medium_flags", "") or "").replace("; ", ", ").strip()
    low_flags = str(row.get("matched_low_flags", "") or "").replace("; ", ", ").strip()
    # Add severity-labelled flag groups
    if high_flags:
        parts.append(f"HIGH, {high_flags}")
    if medium_flags:
        parts.append(f"MEDIUM, {medium_flags}")
    if low_flags:
        parts.append(f"LOW, {low_flags}")
    # Return combined matched flag text
    return " | ".join(parts)

# Extract matched keywords from a domain or text value
def extract_matched_keywords(value: Any) -> str:
    keyword_groups = [
        (MALTA_GOV_FLAGS, "malta_government"),
        (MALTA_FLAGS, "malta_context"),
        (TYPOSQUAT_PATTERNS, "typosquat"),
        (BANK_FLAGS, "brand"),
        (GENERIC_GOV_FLAGS, "generic_gov"),
        (FINANCE_FLAGS, "finance"),
        (ACCOUNT_FLAGS, "account"),
    ]
    # Collect keywords that match the value
    matched: list[str] = []
    for keywords, category in keyword_groups:
        for keyword in keywords:
            if has_keyword_group_match(value, [keyword], category):
                matched.append(str(keyword))
    # Return unique matched keywords
    return "; ".join(dict.fromkeys(matched))

# Define columns for the full review export
FULL_EXPORT = [
    "domain",
    "registered_domain",
    "triage_rank",
    "triage_tier",
    "priority_review_flag",
    "combined_triage_score",
    "model_score",
    "ml_evidence_score",
    "context_evidence_score",
    "crawler_evidence_score",
    "penalty_score",
    "matched_flags_and_keywords",
    "matched_keyword",
    "keyword_risk_flag",
    "triage_explanation",
    "crawler_evidence_flag",
    "credential_form_detected_flag",
    "redirect_detected_flag",
    "matched_high_flags",
    "matched_medium_flags",
    "matched_low_flags",
    "site_status",
    "http_status",
    "final_url_status",
    "form_status",
    "crawler_evidence_strength",
    "page_title",
    "page_meta_description",
    "visible_text_snippet",
    "detected_form_summary",
    "final_url",
]

# Define columns for the analyst scores export
ANALYST_SCORES_EXPORT = [
    "domain",
    "triage_tier",
    "triage_rank",
    "combined_triage_score",
    "model_score",
    "ml_evidence_score",
    "context_evidence_score",
    "crawler_evidence_score",
    "penalty_score",
    "matched_flags_and_keywords",
    "matched_keyword",
    "keyword_risk_flag",
    "triage_explanation",
    "site_status",
    "http_status",
    "final_url_status",
    "form_status",
    "crawler_evidence_strength",
    "page_title",
    "page_meta_description",
    "visible_text_snippet",
    "detected_form_summary",
    "final_url",
]

# Define columns for the compact analyst review export
ANALYST_REVIEW_EXPORT = [
    "domain",
    "triage_tier",
    "triage_rank",
    "matched_keyword",
    "keyword_risk_flag",
    "http_status",
    "site_status",
    "final_url_status",
    "form_status",
    "crawler_evidence_strength",
    "page_title",
    "page_meta_description",
    "visible_text_snippet",
    "final_url",
]

# Define columns for the crawler evidence export
CRAWLER_EXPORT = [
    "domain",
    "site_status",
    "http_status",
    "final_url",
    "redirect_count",
    "page_title",
    "page_meta_description",
    "visible_text_snippet",
    "detected_form_summary",
    "matched_flags_and_keywords",
    "crawler_evidence_flag",
    "redirect_detected_flag",
    "credential_form_detected_flag",
    "dns_has_a",
    "dns_has_mx",
    "tls_handshake_success",
    "html_form_count",
    "html_password_input_count",
]

# Prepare scored domains for analyst-facing review exports
def prepare_review_output(df: pd.DataFrame) -> pd.DataFrame:
    # Work on a copy before formatting output fields
    output_df = df.copy()
    # Rename crawl status to site status when needed
    if "crawl_status" in output_df.columns and "site_status" not in output_df.columns:
        output_df = output_df.rename(columns={"crawl_status": "site_status"})
    # Normalise site status for each row
    output_df["site_status"] = output_df.apply(normalise_site_status_from_row, axis=1)
    # Prepare empty text fallback series
    empty_text = pd.Series("", index=output_df.index)
    # Coerce model score to numeric values
    output_df["model_score"] = pd.to_numeric(output_df.get("model_score", np.nan), errors="coerce")
    # Build matched keywords when missing
    if "matched_keywords" not in output_df.columns:
        output_df["matched_keywords"] = output_df.get("domain", empty_text).map(extract_matched_keywords)
    # Build matched flags and keyword display fields
    output_df["matched_flags"] = output_df.apply(build_matched_flags, axis=1)
    matched_flags_series = output_df["matched_flags"].fillna("").astype(str).str.strip()
    matched_keywords_series = output_df["matched_keywords"].fillna("").astype(str).str.strip()
    output_df["matched_keyword"] = matched_keywords_series.str.split(";").str[0].fillna("").str.strip()
    # Build keyword risk flag from first available severity category
    output_df["keyword_risk_flag"] = ""
    for severity, column in [("HIGH", "matched_high_flags"), ("MEDIUM", "matched_medium_flags"), ("LOW", "matched_low_flags")]:
        raw_series = output_df.get(column, empty_text).fillna("").astype(str).str.replace("; ", ", ", regex=False)
        category_series = raw_series.str.split(",").str[0].fillna("").str.strip()
        fill_mask = output_df["keyword_risk_flag"].eq("") & category_series.ne("")
        output_df.loc[fill_mask, "keyword_risk_flag"] = severity + ", " + category_series[fill_mask]
    # Combine matched flags and keywords into one review field
    output_df["matched_flags_and_keywords"] = np.where(
        matched_flags_series.ne("") & matched_keywords_series.ne(""),
        matched_flags_series + "; " + matched_keywords_series,
        np.where(matched_flags_series.ne(""), matched_flags_series, matched_keywords_series),
    )
    # Format HTTP, final URL, form, crawler evidence, and explanation fields
    output_df["http_status"] = output_df.get("http_status", empty_text).map(normalise_http_status_value)
    output_df["final_url_status"] = output_df.get("final_url", empty_text).fillna("").astype(str).str.strip().ne("").map({True: "final URL available", False: "final URL unavailable"})
    output_df["form_status"] = output_df.apply(build_form_status_from_row, axis=1)
    output_df["detected_form_summary"] = output_df["form_status"]
    output_df["crawler_evidence_strength"] = output_df.apply(build_crawler_evidence_clause, axis=1)
    output_df["triage_explanation"] = output_df.apply(triage_explanation, axis=1)
    # Fill text output columns with empty strings
    for column in [
        "matched_high_flags",
        "matched_medium_flags",
        "matched_low_flags",
        "matched_flags",
        "matched_keyword",
        "keyword_risk_flag",
        "matched_keywords",
        "matched_flags_and_keywords",
        "triage_explanation",
        "http_status",
        "final_url_status",
        "form_status",
        "crawler_evidence_strength",
        "page_title",
        "page_meta_description",
        "visible_text_snippet",
        "detected_form_summary",
    ]:
        if column in output_df.columns:
            output_df[column] = output_df[column].fillna("").astype(str)
    # Return the formatted review dataframe
    return output_df

# Build the full, analyst score, and compact analyst review outputs
def build_review_outputs(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Prepare common review fields
    review_df = prepare_review_output(df)
    # Build full review export with available configured columns
    full_review = review_df[[column for column in FULL_EXPORT if column in review_df.columns]].copy()
    # Keep priority review or high and medium matched rows for analyst exports
    analyst_base = review_df[numeric_series(review_df, "priority_review_flag").eq(1) | matched_high_medium_mask(review_df)].copy()
    # Build analyst score and compact analyst review exports
    analyst_scores = analyst_base[[column for column in ANALYST_SCORES_EXPORT if column in analyst_base.columns]].copy()
    analyst_review = analyst_base[[column for column in ANALYST_REVIEW_EXPORT if column in analyst_base.columns]].copy()
    # Return all review export dataframes
    return full_review, analyst_scores, analyst_review

In [ ]:
# Print a timestamped progress message for long-running notebook stages
def report_progress(stage: str, detail: str = "", start_time: float | None = None) -> None:
    # Build a UTC timestamp for the progress message
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    # Start the message with the stage name
    message = f"[{timestamp}] {stage}"
    # Add optional detail text
    if detail:
        message += f" - {detail}"
    # Add elapsed time when a start time is supplied
    if start_time is not None:
        message += f" ({time.perf_counter() - start_time:.2f}s)"
    # Print the progress message
    print(message)

# Round floating-point columns for cleaner output files
def round_output_numeric_columns(df: pd.DataFrame, digits: int = 3) -> pd.DataFrame:
    # Work on a copy to preserve the original dataframe
    rounded = df.copy()
    # Select floating-point columns
    float_columns = rounded.select_dtypes(include=["floating", "float16", "float32", "float64"]).columns
    # Round floating-point values when present
    if len(float_columns) > 0:
        rounded.loc[:, float_columns] = rounded.loc[:, float_columns].round(digits)
    # Return the rounded dataframe
    return rounded

# Escape text that spreadsheet tools could interpret as formulas
def escape_spreadsheet_formula_text(value: Any) -> Any:
    # Leave non-text values unchanged
    if not isinstance(value, str):
        return value
    # Leave empty text unchanged
    if not value:
        return value
    # Prefix formula-like text with a tab to prevent spreadsheet execution
    if value[0] in ("=", "+", "-", "@"):
        return "\t" + value
    # Return safe text unchanged
    return value

# Prepare a dataframe for safe CSV export
def prepare_csv_output(df: pd.DataFrame, digits: int = 3) -> pd.DataFrame:
    # Round numeric columns first
    output_df = round_output_numeric_columns(df, digits=digits)
    # Select text columns for formula escaping
    object_columns = output_df.select_dtypes(include=["object", "string"]).columns
    # Escape formula-like text values
    for column in object_columns:
        output_df.loc[:, column] = output_df[column].map(escape_spreadsheet_formula_text)
    # Return CSV-ready dataframe
    return output_df

# Build crawler results output for enriched domains
def build_crawler_results_df(
    df: pd.DataFrame,
    input_name: str,
    crawled_timestamp: str,
) -> pd.DataFrame:
    # Keep rows where live enrichment was attempted and prepare review fields
    enrichment_df = prepare_review_output(
        df[numeric_series(df, "enrichment_attempted").eq(1)].copy()
    )
    # Add input file and crawl timestamp metadata
    enrichment_df["whoisds_input_file"] = input_name
    enrichment_df["crawled_timestamp"] = crawled_timestamp
    # Return available crawler export columns
    return enrichment_df[[column for column in (["whoisds_input_file", "crawled_timestamp"] + CRAWLER_EXPORT) if column in enrichment_df.columns]].copy()

# Write a CSV output file and report progress
def write_csv_output(df: pd.DataFrame, path: Path, stage: str) -> None:
    # Capture write start time
    start = time.perf_counter()
    # Prepare and write the CSV with UTF-8 BOM for spreadsheet compatibility
    prepare_csv_output(df).to_csv(path, index=False, float_format="%.3f", encoding="utf-8-sig")
    # Report the write result
    report_progress(stage, f"{len(df):,} rows -> {path.name}", start)

# Define output paths for review and crawler files
OUTPUT_PATHS = {
    "full_review": WHOISDS_DEBUGGING_OUTPUT_DIR / "whoisds_full_review.csv",
    "analyst_scores": WHOISDS_DEBUGGING_OUTPUT_DIR / "whoisds_analyst_scores.csv",
    "analyst_review": WHOISDS_RUN_OUTPUT_DIR / "whoisds_analyst_review.csv",
    "crawler_results": WHOISDS_DEBUGGING_OUTPUT_DIR / "whoisds_crawler_results.csv",
}

In [ ]:
# Prepare the matched-flags crawl queue
report_progress("crawler_queue_started", "Preparing High/Medium matched-flags crawl queue")
operational_scored_df = whoisds_enriched.copy()
operational_scored_df = prepare_matched_flag_queue(operational_scored_df)

# Keep only High or Medium matched-flag domains for selective enrichment and scoring
operational_scored_df = operational_scored_df[
    matched_high_medium_mask(operational_scored_df)
].copy().reset_index(drop=True)

# Run selective enrichment on the matched-flags queue
report_progress("selective_enrichment_started", "Crawling High/Medium matched-flags queue")
operational_scored_df = selective_enrich_prioritised_domains(operational_scored_df)

# Score enriched domains with the selected XGBoost model
report_progress("model_scoring_started", "Scoring enriched matched-flags queue")
operational_scored_df = score_domains_with_model(operational_scored_df)

# Apply operational evidence-combination triage
report_progress("triage_scoring_started", "Applying evidence-combination triage to matched-flags queue")
operational_scored_df, diagnostics = apply_operational_scoring(
    operational_scored_df,
    probability_column="model_score",
)

# Build and display SHAP summaries for live-batch sanity checks
report_progress("shap_summary_started", "Rendering SHAP summary for model debugging and live-batch sanity checks")
live_shap_feature_summary_df, live_shap_top_features_df = build_live_scoring_shap_outputs(operational_scored_df)
if not live_shap_feature_summary_df.empty:
    display(live_shap_feature_summary_df)
if not live_shap_top_features_df.empty:
    display(live_shap_top_features_df)

# Build analyst review and crawler result outputs
full_review_df, analyst_scores_df, analyst_review_df = build_review_outputs(operational_scored_df)
current_input = WHOISDS_SOURCE_PATH.name
current_crawled_timestamp = CRAWLED_TIMESTAMP
crawler_results_df = build_crawler_results_df(
    operational_scored_df,
    current_input,
    current_crawled_timestamp,
)

# Write output CSV files
write_csv_output(full_review_df, OUTPUT_PATHS["full_review"], "full_review_written")
write_csv_output(analyst_scores_df, OUTPUT_PATHS["analyst_scores"], "analyst_scores_written")
write_csv_output(analyst_review_df, OUTPUT_PATHS["analyst_review"], "analyst_review_written")
write_csv_output(crawler_results_df, OUTPUT_PATHS["crawler_results"], "crawler_results_written")

# Build blacklist metric rows for the summary table
blacklist_metric_rows = [f"Blacklist hits ({keyword})" for keyword in BLACKLIST_KEYWORDS]
blacklist_metric_values = [int(BLACKLIST_HIT_COUNTS.get(keyword, 0)) for keyword in BLACKLIST_KEYWORDS]

# Build the run summary table
summary_df = pd.DataFrame(
    {
        "metric": [
            "High tier",
            "Medium tier",
            "Low tier",
            "Crawled domains",
            "Blacklist removed rows",
        ] + blacklist_metric_rows + [
            "Crawler evidence rows",
            "Credential form rows",
            "Redirect rows",
        ],
        "value": [
            diagnostics["high_tier_count"],
            diagnostics["medium_tier_count"],
            diagnostics["low_tier_count"],
            diagnostics["crawl_candidate_count"],
            BLACKLIST_REMOVED_COUNT,
        ] + blacklist_metric_values + [
            diagnostics["crawler_evidence_count"],
            diagnostics["credential_form_count"],
            diagnostics["redirect_detected_count"],
        ],
    }
)

# Build triage tier distribution counts
triage_distribution_df = (
    operational_scored_df["triage_tier"]
    .replace({"High": "High triage", "Medium": "Medium triage"})
    .value_counts(dropna=False)
    .rename_axis("triage_tier")
    .reset_index(name="count")
)

# Count rows with High and Medium matched flags
matched_high_series = operational_scored_df.get("matched_high_flags", pd.Series("", index=operational_scored_df.index)).fillna("").astype(str).str.strip()
matched_medium_series = operational_scored_df.get("matched_medium_flags", pd.Series("", index=operational_scored_df.index)).fillna("").astype(str).str.strip()

# Add matched-flag counts to the triage distribution table
triage_distribution_df = pd.concat(
    [
        triage_distribution_df,
        pd.DataFrame(
            {
                "triage_tier": ["High flags", "Medium flags"],
                "count": [
                    int(matched_high_series.ne("").sum()),
                    int(matched_medium_series.ne("").sum()),
                ],
            }
        ),
    ],
    ignore_index=True,
)

# Build descriptive statistics for combined triage scores
score_stats_df = (
    operational_scored_df["combined_triage_score"]
    .describe()
    .rename_axis("statistic")
    .reset_index(name="value")
)

# Print final output row counts
print(f"Full review rows     : {len(full_review_df):,}")
print(f"Analyst scores rows  : {len(analyst_scores_df):,}")
print(f"Analyst review rows  : {len(analyst_review_df):,}")
print(f"Crawler rows         : {len(crawler_results_df):,}")

# Display summary, triage distribution, and score statistics
display(summary_df)
display(triage_distribution_df)
display(score_stats_df)